# Domainrouting

## Domainoverview

In [1]:
import pandas as pd

# TSV einlesen (so ähnlich machst du es ja schon)
df = pd.read_csv(
    "../2_Finetuning_Masterthesis/LIAR/preprocessed_train_cleaned_binary.csv",
    sep="\t",
    quotechar='"',
    engine="python",
    dtype=str,
)

# 1) Alle Subjects in einzelne Domains aufsplitten
domains_series = (
    df["subjects"]
    .dropna()                     # NaNs raus
    .astype(str)                  # zur Sicherheit als String
    .str.split(",")               # in Listen splitten
    .explode()                    # jede Domain in eigene Zeile
    .str.strip()                  # Whitespace trimmen
)

# 2) Einzigartige Domains (alphabetisch sortiert)
unique_domains = sorted(domains_series.unique())
print("Einzigartige Domains:")
for d in unique_domains:
    print("-", d)

# 3) Häufigkeiten pro einzelner Domain
domain_counts = domains_series.value_counts().sort_index()
print("\nHäufigkeiten pro Domain:")
print(domain_counts)


Einzigartige Domains:
- 10-news-tampa-bay
- Alcohol
- abc-news-week
- abortion
- afghanistan
- after-the-fact
- agriculture
- animals
- autism
- bankruptcy
- baseball
- bipartisanship
- bush-administration
- campaign-advertising
- campaign-finance
- candidates-biography
- cap-and-trade
- census
- children
- china
- city-budget
- city-government
- civil-rights
- climate-change
- colbert-report
- congress
- congressional-rules
- consumer-safety
- corporations
- corrections-and-updates
- county-budget
- county-government
- crime
- criminal-justice
- death-penalty
- debates
- debt
- deficit
- disability
- diversity
- drugs
- ebola
- economy
- education
- elections
- energy
- environment
- ethics
- fake-news
- families
- federal-budget
- financial-regulation
- fires
- florida
- florida-amendments
- food
- food-safety
- foreign-policy
- gambling
- gas-prices
- gays-and-lesbians
- government-efficiency
- government-regulation
- guns
- health-care
- history
- homeland-security
- homeless
- hou

## 8 Superlabels

In [1]:
domain_to_super_8 = {
    # ----- 1. economy -----
    "Alcohol": "economy",
    "agriculture": "economy",
    "bankruptcy": "economy",
    "corporations": "economy",
    "consumer-safety": "economy",
    "debt": "economy",
    "deficit": "economy",
    "economy": "economy",
    "financial-regulation": "economy",
    "gas-prices": "economy",
    "gambling": "economy",
    "housing": "economy",
    "income": "economy",
    "infrastructure": "economy",
    "job-accomplishments": "economy",
    "jobs": "economy",
    "labor": "economy",
    "market-regulation": "economy",
    "medicaid": "economy",
    "pensions": "economy",
    "poverty": "economy",
    "retirement": "economy",
    "small-business": "economy",
    "social-security": "economy",
    "stimulus": "economy",
    "taxes": "economy",
    "tourism": "economy",
    "trade": "economy",
    "unions": "economy",
    "wealth": "economy",
    "workers": "economy",
    "food": "economy",

    # ----- 2. health_social -----
    "health-care": "health_social",
    "medicare": "health_social",
    "public-health": "health_social",
    "food-safety": "health_social",
    "drugs": "health_social",
    "marijuana": "health_social",
    "ebola": "health_social",
    "hunger": "health_social",
    "disability": "health_social",
    "women": "health_social",
    "families": "health_social",
    "sexuality": "health_social",
    "veterans": "health_social",
    "children": "health_social",
    "autism": "health_social",
    "homeless": "health_social",

    # ----- 3. foreign_security -----
    "afghanistan": "foreign_security",
    "foreign-policy": "foreign_security",
    "iraq": "foreign_security",
    "israel": "foreign_security",
    "china": "foreign_security",
    "military": "foreign_security",
    "homeland-security": "foreign_security",
    "terrorism": "foreign_security",
    "nuclear": "foreign_security",
    "patriotism": "foreign_security",
    "natural-disasters": "foreign_security",
    "public-safety": "foreign_security",

    # ----- 4. law_rights -----
    "abortion": "law_rights",
    "civil-rights": "law_rights",
    "crime": "law_rights",
    "criminal-justice": "law_rights",
    "guns": "law_rights",
    "legal-issues": "law_rights",
    "privacy": "law_rights",
    "human-rights": "law_rights",
    "supreme-court": "law_rights",
    "kagan-nomination": "law_rights",
    "sotomayor-nomination": "law_rights",
    "islam": "law_rights",
    "immigration": "law_rights",

    # ----- 5. politics_government -----
    "bipartisanship": "politics_government",
    "bush-administration": "politics_government",
    "campaign-advertising": "politics_government",
    "campaign-finance": "politics_government",
    "candidates-biography": "politics_government",
    "congress": "politics_government",
    "congressional-rules": "politics_government",
    "corrections-and-updates": "politics_government",
    "county-government": "politics_government",
    "county-budget": "politics_government",
    "city-government": "politics_government",
    "city-budget": "politics_government",
    "debates": "politics_government",
    "elections": "politics_government",
    "ethics": "politics_government",
    "fake-news": "politics_government",
    "government-efficiency": "politics_government",
    "government-regulation": "politics_government",
    "message-machine": "politics_government",
    "message-machine-2012": "politics_government",
    "message-machine-2014": "politics_government",
    "new-hampshire-2012": "politics_government",
    "occupy-wall-street": "politics_government",
    "politifacts-top-promises": "politics_government",
    "polls": "politics_government",
    "public-service": "politics_government",
    "redistricting": "politics_government",
    "transparency": "politics_government",
    "states": "politics_government",
    "state-budget": "politics_government",
    "state-finances": "politics_government",
    "voting-record": "politics_government",

    # ----- 6. environment_energy -----
    "cap-and-trade": "environment_energy",
    "climate-change": "environment_energy",
    "environment": "environment_energy",
    "energy": "environment_energy",
    "oil-spill": "environment_energy",
    "water": "environment_energy",
    "weather": "environment_energy",
    "transportation": "environment_energy",
    "urban": "environment_energy",
    "recreation": "environment_energy",

    # ----- 7. society_culture -----
    "education": "society_culture",
    "gays-and-lesbians": "society_culture",
    "diversity": "society_culture",
    "religion": "society_culture",
    "marriage": "society_culture",
    "population": "society_culture",
    "pop-culture": "society_culture",
    "sports": "society_culture",
    "nightlife": "society_culture",
    "history": "society_culture",

    # ----- 8. misc -----
    "abc-news-week": "misc",
    "pundits": "misc",
    "obama-birth-certificate": "misc",
    "space": "misc",
    "technology": "misc",
    "10-news-tampa-bay": "misc",
    "after-the-fact": "misc",
    "lottery": "misc",
}

### Train Domain Routing Model

In [2]:
from train_domain_expert_domain_config import DomainConfig
from train_domain_expert_domain_classifier import train_domain_classifier, evaluate_on_testset


domain_config = DomainConfig(
    domain_to_super=domain_to_super_8,
    # Falls du die Reihenfolge explizit festlegen willst:
    super_labels=[
        "economy",
        "health_social",
        "foreign_security",
        "law_rights",
        "politics_government",
        "environment_energy",
        "society_culture",
        "misc",
    ]
)

# ========= 2) Pfade & Modell-ID =========

base_model_id = "meta-llama/Llama-3.1-8B-Instruct"
train_path = "../2_Finetuning_Masterthesis/LIAR/preprocessed_train_cleaned_binary.csv"
test_path = "../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"
output_dir = "adapters/llama-liar-statement-domain-lora_8Classes"

# ========= 3) Training starten =========

if __name__ == "__main__":
    # Train
    train_domain_classifier(
        train_path=train_path,
        domain_config=domain_config,
        base_model_id=base_model_id,
        output_dir=output_dir,
        num_train_epochs=3,
        learning_rate=5e-4,
        per_device_train_batch_size=4,
        max_length=256,
    )

    # Eval
    evaluate_on_testset(
        test_path=test_path,
        domain_config=domain_config,
        base_model_id=base_model_id,
        lora_dir=output_dir,
    )


Super label distribution (train):
super_domain
economy                3809
politics_government    1698
health_social          1402
law_rights             1276
foreign_security        971
society_culture         473
environment_energy      412
misc                    199
Name: count, dtype: int64


Map:   0%|          | 0/10240 [00:00<?, ? examples/s]

Example tokenized sample keys: dict_keys(['input_ids', 'attention_mask'])
Input length: 156


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

...

Step,Training Loss
50,0.663900
100,0.406500
150,0.397100
200,0.389700
250,0.389700
300,0.395000
350,0.387500
400,0.395700
450,0.396500
500,0.406800


✅ LoRA adapter & tokenizer saved to: adapters/llama-liar-statement-domain-lora_8Classes
Test set size: 1267
True label distribution (test):
super_domain_true
economy                473
politics_government    233
law_rights             145
health_social          143
foreign_security       122
society_culture         61
environment_energy      56
misc                    34
Name: count, dtype: int64


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🧠 Routing statements: 100%|██████████| 1267/1267 [09:57<00:00,  2.12it/s]

Predicted label distribution (test):
super_domain_pred
economy                510
politics_government    208
health_social          158
law_rights             147
foreign_security       113
society_culture         59
environment_energy      52
misc                    20
Name: count, dtype: int64

Accuracy (super_domain) on test set: 0.734

Classification report (per superlabel):
                     precision    recall  f1-score   support

            economy       0.77      0.83      0.80       473
      health_social       0.66      0.73      0.70       143
   foreign_security       0.84      0.78      0.81       122
         law_rights       0.73      0.74      0.73       145
politics_government       0.69      0.62      0.65       233
 environment_energy       0.71      0.66      0.69        56
    society_culture       0.64      0.62      0.63        61
               misc       0.60      0.35      0.44        34

           accuracy                           0.73      1267
      

In [3]:
from train_domain_expert_domain_config import DomainConfig
from train_domain_expert_domain_classifier import train_domain_classifier, evaluate_on_testset


domain_config = DomainConfig(
    domain_to_super=domain_to_super_8,
    # Falls du die Reihenfolge explizit festlegen willst:
    super_labels=[
        "economy",
        "health_social",
        "foreign_security",
        "law_rights",
        "politics_government",
        "environment_energy",
        "society_culture",
        "misc",
    ]
)

# ========= 2) Pfade & Modell-ID =========

base_model_id = "google/gemma-7b-it"
train_path = "../2_Finetuning_Masterthesis/LIAR/preprocessed_train_cleaned_binary.csv"
test_path = "../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"
output_dir = "adapters/gemma-liar-statement-domain-lora_8Classes"

# ========= 3) Training starten =========

if __name__ == "__main__":
    # Train
    train_domain_classifier(
        train_path=train_path,
        domain_config=domain_config,
        base_model_id=base_model_id,
        output_dir=output_dir,
        num_train_epochs=3,
        learning_rate=5e-4,
        per_device_train_batch_size=4,
        max_length=256,
    )

    # Eval
    evaluate_on_testset(
        test_path=test_path,
        domain_config=domain_config,
        base_model_id=base_model_id,
        lora_dir=output_dir,
    )


Super label distribution (train):
super_domain
economy                3809
politics_government    1698
health_social          1402
law_rights             1276
foreign_security        971
society_culture         473
environment_energy      412
misc                    199
Name: count, dtype: int64


Map:   0%|          | 0/10240 [00:00<?, ? examples/s]

Example tokenized sample keys: dict_keys(['input_ids', 'attention_mask'])
Input length: 141


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

...

Step,Training Loss
50,1.389100
100,0.723400
150,0.596200
200,0.520200
250,0.496400
300,0.498700
350,0.491600
400,0.509100
450,0.611300
500,0.609100


✅ LoRA adapter & tokenizer saved to: adapters/gemma-liar-statement-domain-lora_8Classes
Test set size: 1267
True label distribution (test):
super_domain_true
economy                473
politics_government    233
law_rights             145
health_social          143
foreign_security       122
society_culture         61
environment_energy      56
misc                    34
Name: count, dtype: int64


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🧠 Routing statements: 100%|██████████| 1267/1267 [03:42<00:00,  5.68it/s]


Predicted label distribution (test):
super_domain_pred
economy                496
health_social          182
law_rights             177
politics_government    145
foreign_security       117
environment_energy      71
society_culture         69
misc                    10
Name: count, dtype: int64

Accuracy (super_domain) on test set: 0.725

Classification report (per superlabel):
                     precision    recall  f1-score   support

            economy       0.78      0.81      0.79       473
      health_social       0.65      0.83      0.73       143
   foreign_security       0.82      0.79      0.80       122
         law_rights       0.67      0.82      0.74       145
politics_government       0.77      0.48      0.59       233
 environment_energy       0.58      0.73      0.65        56
    society_culture       0.59      0.67      0.63        61
               misc       0.60      0.18      0.27        34

           accuracy                           0.73      1267
      

In [3]:
from train_domain_expert_domain_config import DomainConfig
from train_domain_expert_domain_classifier import train_domain_classifier, evaluate_on_testset


domain_config = DomainConfig(
    domain_to_super=domain_to_super_8,
    # Falls du die Reihenfolge explizit festlegen willst:
    super_labels=[
        "economy",
        "health_social",
        "foreign_security",
        "law_rights",
        "politics_government",
        "environment_energy",
        "society_culture",
        "misc",
    ]
)

# ========= 2) Pfade & Modell-ID =========

base_model_id = "deepseek-ai/deepseek-llm-7b-base"
train_path = "../2_Finetuning_Masterthesis/LIAR/preprocessed_train_cleaned_binary.csv"
test_path = "../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"
output_dir = "adapters/deepseek-liar-statement-domain-lora_8Classes"

# ========= 3) Training starten =========

if __name__ == "__main__":
    # Train
    train_domain_classifier(
        train_path=train_path,
        domain_config=domain_config,
        base_model_id=base_model_id,
        output_dir=output_dir,
        num_train_epochs=3,
        learning_rate=5e-4,
        per_device_train_batch_size=4,
        max_length=256,
    )

    # Eval
    evaluate_on_testset(
        test_path=test_path,
        domain_config=domain_config,
        base_model_id=base_model_id,
        lora_dir=output_dir,
    )


Super label distribution (train):
super_domain
economy                3809
politics_government    1698
health_social          1402
law_rights             1276
foreign_security        971
society_culture         473
environment_energy      412
misc                    199
Name: count, dtype: int64


Map:   0%|          | 0/10240 [00:00<?, ? examples/s]

Example tokenized sample keys: dict_keys(['input_ids', 'attention_mask'])
Input length: 147


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 18,739,200 || all params: 6,929,104,896 || trainable%: 0.2704


...

Step,Training Loss
50,0.599000
100,0.422300
150,0.410400
200,0.395700
250,0.395000
300,0.401500
350,0.391800
400,0.396600
450,0.398600
500,0.410100


✅ LoRA adapter & tokenizer saved to: adapters/deepseek-liar-statement-domain-lora_8Classes
Test set size: 1267
True label distribution (test):
super_domain_true
economy                473
politics_government    233
law_rights             145
health_social          143
foreign_security       122
society_culture         61
environment_energy      56
misc                    34
Name: count, dtype: int64


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🧠 Routing statements: 100%|██████████| 1267/1267 [09:25<00:00,  2.24it/s]

Predicted label distribution (test):
super_domain_pred
economy                482
politics_government    232
health_social          162
law_rights             151
foreign_security       121
society_culture         57
environment_energy      48
misc                    14
Name: count, dtype: int64

Accuracy (super_domain) on test set: 0.744

Classification report (per superlabel):
                     precision    recall  f1-score   support

            economy       0.80      0.82      0.81       473
      health_social       0.71      0.80      0.75       143
   foreign_security       0.79      0.79      0.79       122
         law_rights       0.72      0.75      0.74       145
politics_government       0.69      0.68      0.68       233
 environment_energy       0.71      0.61      0.65        56
    society_culture       0.60      0.56      0.58        61
               misc       0.57      0.24      0.33        34

           accuracy                           0.74      1267
      

### Train-Experts

In [4]:
import pandas as pd
SUPER_LABELS = [
        "economy",
        "health_social",
        "foreign_security",
        "law_rights",
        "politics_government",
        "environment_energy",
        "society_culture",
        "misc",
    ]
def map_subjects_to_super(subjects_str: str) -> str:
    """
    Map LIAR 'subjects' (comma-separated topics) to exactly one super label.
    """
    if not isinstance(subjects_str, str) or not subjects_str.strip():
        return "misc"

    domains = [s.strip() for s in subjects_str.split(",") if s.strip()]
    super_labels = {domain_to_super_8.get(d, "misc") for d in domains}

    # priorisierte Reihenfolge: erstes vorkommendes Label aus SUPER_LABELS
    for label in SUPER_LABELS:
        if label in super_labels:
            return label
    return "misc"

data_path = "../2_Finetuning_Masterthesis/LIAR/preprocessed_train_cleaned_binary.csv"  # anpassen falls nötig

df_all = pd.read_csv(
    data_path,
    sep="\t",
    quotechar='"',
    engine="python",
    dtype=str,
)

# super_domain aus subjects erzeugen
df_all["super_domain"] = df_all["subjects"].apply(map_subjects_to_super)

# Aufräumen
df_all = df_all.dropna(subset=["statement", "label", "super_domain"])
df_all = df_all[df_all["statement"].str.strip() != ""]
df_all = df_all[df_all["label"].isin(["True", "False"])]

print("Super-Domain-Verteilung:")
print(df_all["super_domain"].value_counts())


Super-Domain-Verteilung:
super_domain
economy                3809
politics_government    1698
health_social          1402
law_rights             1276
foreign_security        971
society_culture         473
environment_energy      412
misc                    199
Name: count, dtype: int64


In [6]:
df_all.to_csv("Results/preprocessed_train_cleaned_binary_with_super_domain_8.csv", sep="\t", index=False)


In [7]:
import pandas as pd

data_path = "Results/preprocessed_train_cleaned_binary_with_super_domain_8.csv"

df_all = pd.read_csv(
    data_path,
    sep="\t",
    quotechar='"',
    engine="python",
    dtype=str,
)

# Sicherstellen, dass wir nur sinnvolle Zeilen haben
df_all = df_all.dropna(subset=["statement", "label", "super_domain"])
df_all = df_all[df_all["statement"].str.strip() != ""]
df_all = df_all[df_all["label"].isin(["True", "False"])]

print("Gesamtverteilung super_domain:")
print(df_all["super_domain"].value_counts())

SUPER_LABELS = [
        "economy",
        "health_social",
        "foreign_security",
        "law_rights",
        "politics_government",
        "environment_energy",
        "society_culture",
        "misc",
]

# Pro Domain ein DataFrame
domain_dfs = {
    domain: df_all[df_all["super_domain"] == domain].reset_index(drop=True)
    for domain in SUPER_LABELS
}

for dom, df_dom in domain_dfs.items():
    print(f"\nDomain: {dom} -> {len(df_dom)} Beispiele")


Gesamtverteilung super_domain:
super_domain
economy                3809
politics_government    1698
health_social          1402
law_rights             1276
foreign_security        971
society_culture         473
environment_energy      412
misc                    199
Name: count, dtype: int64

Domain: economy -> 3809 Beispiele

Domain: health_social -> 1402 Beispiele

Domain: foreign_security -> 971 Beispiele

Domain: law_rights -> 1276 Beispiele

Domain: politics_government -> 1698 Beispiele

Domain: environment_energy -> 412 Beispiele

Domain: society_culture -> 473 Beispiele

Domain: misc -> 199 Beispiele


In [8]:
# run_train_experts.py (oder Notebook-Zelle)
import sys, os
import pandas as pd

# falls du in Jupyter bist:
sys.path.append(os.getcwd())

from train_expert_expert_config import ExpertConfig
from train_expert_expert_trainer import train_all_experts

# 1) Trainingsdaten laden – EIN gemeinsamer DataFrame
train_path = "Results/preprocessed_train_cleaned_binary_with_super_domain_8.csv"

df_train = pd.read_csv(
    train_path,
    sep="\t",
    quotechar='"',
    engine="python",
    dtype=str,
)

# 2) Konfiguration der Experten (Superdomains + Spaltennamen)
config = ExpertConfig(
    super_domains=[
        "economy",
        "health_social",
        "foreign_security",
        "law_rights",
        "politics_government",
        "environment_energy",
        "society_culture",
        "misc",
    ],
    base_model_id="meta-llama/Llama-3.1-8B-Instruct",
    text_column="statement",
    label_column="label",
    domain_column="super_domain",
    max_length=256,
    num_train_epochs=3,
    learning_rate=5e-4,
    per_device_train_batch_size=4,
)

# 3) Output-Root – dort bekommt jede Domain seinen eigenen Unterordner
output_root = "adapters/experts_8Classes"

saved_paths = train_all_experts(
    df=df_train,
    config=config,
    output_root=output_root,
)

print("\nFertig! Modelle gespeichert unter:")
for dom, p in saved_paths.items():
    print(f"  {dom}: {p}")


2026-02-01 13:48:29.398978: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.



=== Training expert for domain: economy ===
Samples: 3809


Map:   0%|          | 0/3809 [00:00<?, ? examples/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 20,971,520 || all params: 8,051,232,768 || trainable%: 0.2605


🚀 Training:   0%|          | 0/75 [00:00<?, ?it/s]

Step,Training Loss
25,0.842300
50,0.323000
75,0.206300


✔️ Expert-LoRA for 'misc' saved to adapters/experts_8Classes/misc

Fertig! Modelle gespeichert unter:
  economy: adapters/experts_8Classes/economy
  health_social: adapters/experts_8Classes/health_social
  foreign_security: adapters/experts_8Classes/foreign_security
  law_rights: adapters/experts_8Classes/law_rights
  politics_government: adapters/experts_8Classes/politics_government
  environment_energy: adapters/experts_8Classes/environment_energy
  society_culture: adapters/experts_8Classes/society_culture
  misc: adapters/experts_8Classes/misc


In [9]:
# run_train_experts.py (oder Notebook-Zelle)
import sys, os
import pandas as pd

# falls du in Jupyter bist:
sys.path.append(os.getcwd())

from train_expert_expert_config import ExpertConfig
from train_expert_expert_trainer import train_all_experts

# 1) Trainingsdaten laden – EIN gemeinsamer DataFrame
train_path = "Results/preprocessed_train_cleaned_binary_with_super_domain_8.csv"

df_train = pd.read_csv(
    train_path,
    sep="\t",
    quotechar='"',
    engine="python",
    dtype=str,
)

# 2) Konfiguration der Experten (Superdomains + Spaltennamen)
config = ExpertConfig(
    super_domains=[
        "economy",
        "health_social",
        "foreign_security",
        "law_rights",
        "politics_government",
        "environment_energy",
        "society_culture",
        "misc",
    ],
    base_model_id="google/gemma-7b-it",
    text_column="statement",
    label_column="label",
    domain_column="super_domain",
    max_length=256,
    num_train_epochs=3,
    learning_rate=5e-4,
    per_device_train_batch_size=4,
)

# 3) Output-Root – dort bekommt jede Domain seinen eigenen Unterordner
output_root = "adapters/gemma_experts_8Classes"

saved_paths = train_all_experts(
    df=df_train,
    config=config,
    output_root=output_root,
)

print("\nFertig! Modelle gespeichert unter:")
for dom, p in saved_paths.items():
    print(f"  {dom}: {p}")



=== Training expert for domain: economy ===
Samples: 3809


Map:   0%|          | 0/3809 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 25,001,984 || all params: 8,562,682,880 || trainable%: 0.2920


🚀 Training:   0%|          | 0/1431 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 25,001,984 || all params: 8,562,682,880 || trainable%: 0.2920


🚀 Training:   0%|          | 0/75 [00:00<?, ?it/s]

Step,Training Loss
25,2.506700
50,0.605600
75,0.410600


✔️ Expert-LoRA for 'misc' saved to adapters/gemma_experts_8Classes/misc

Fertig! Modelle gespeichert unter:
  economy: adapters/gemma_experts_8Classes/economy
  health_social: adapters/gemma_experts_8Classes/health_social
  foreign_security: adapters/gemma_experts_8Classes/foreign_security
  law_rights: adapters/gemma_experts_8Classes/law_rights
  politics_government: adapters/gemma_experts_8Classes/politics_government
  environment_energy: adapters/gemma_experts_8Classes/environment_energy
  society_culture: adapters/gemma_experts_8Classes/society_culture
  misc: adapters/gemma_experts_8Classes/misc


In [10]:
# run_train_experts.py (oder Notebook-Zelle)
import sys, os
import pandas as pd

# falls du in Jupyter bist:
sys.path.append(os.getcwd())

from train_expert_expert_config import ExpertConfig
from train_expert_expert_trainer import train_all_experts

# 1) Trainingsdaten laden – EIN gemeinsamer DataFrame
train_path = "Results/preprocessed_train_cleaned_binary_with_super_domain_8.csv"

df_train = pd.read_csv(
    train_path,
    sep="\t",
    quotechar='"',
    engine="python",
    dtype=str,
)

# 2) Konfiguration der Experten (Superdomains + Spaltennamen)
config = ExpertConfig(
    super_domains=[
        "economy",
        "health_social",
        "foreign_security",
        "law_rights",
        "politics_government",
        "environment_energy",
        "society_culture",
        "misc",
    ],
    base_model_id="deepseek-ai/deepseek-llm-7b-base",
    text_column="statement",
    label_column="label",
    domain_column="super_domain",
    max_length=256,
    num_train_epochs=3,
    learning_rate=5e-4,
    per_device_train_batch_size=4,
)

# 3) Output-Root – dort bekommt jede Domain seinen eigenen Unterordner
output_root = "adapters/deepseek_experts_8Classes"

saved_paths = train_all_experts(
    df=df_train,
    config=config,
    output_root=output_root,
)

print("\nFertig! Modelle gespeichert unter:")
for dom, p in saved_paths.items():
    print(f"  {dom}: {p}")



=== Training expert for domain: economy ===
Samples: 3809


Map:   0%|          | 0/3809 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 18,739,200 || all params: 6,929,104,896 || trainable%: 0.2704


🚀 Training:   0%|          | 0/1431 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 18,739,200 || all params: 6,929,104,896 || trainable%: 0.2704


🚀 Training:   0%|          | 0/75 [00:00<?, ?it/s]

Step,Training Loss
25,0.742100
50,0.388300
75,0.342900


✔️ Expert-LoRA for 'misc' saved to adapters/deepseek_experts_8Classes/misc

Fertig! Modelle gespeichert unter:
  economy: adapters/deepseek_experts_8Classes/economy
  health_social: adapters/deepseek_experts_8Classes/health_social
  foreign_security: adapters/deepseek_experts_8Classes/foreign_security
  law_rights: adapters/deepseek_experts_8Classes/law_rights
  politics_government: adapters/deepseek_experts_8Classes/politics_government
  environment_energy: adapters/deepseek_experts_8Classes/environment_energy
  society_culture: adapters/deepseek_experts_8Classes/society_culture
  misc: adapters/deepseek_experts_8Classes/misc


### Evaluation

In [2]:
import pandas as pd
SUPER_LABELS = [
        "economy",
        "health_social",
        "foreign_security",
        "law_rights",
        "politics_government",
        "environment_energy",
        "society_culture",
        "misc",
]
def map_subjects_to_super(subjects_str: str) -> str:
    """
    Map LIAR 'subjects' (comma-separated topics) to exactly one super label.
    """
    if not isinstance(subjects_str, str) or not subjects_str.strip():
        return "misc"

    domains = [s.strip() for s in subjects_str.split(",") if s.strip()]
    super_labels = {domain_to_super_8.get(d, "misc") for d in domains}

    # priorisierte Reihenfolge: erstes vorkommendes Label aus SUPER_LABELS
    for label in SUPER_LABELS:
        if label in super_labels:
            return label
    return "misc"

data_path = "../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"  # anpassen falls nötig

df_all = pd.read_csv(
    data_path,
    sep="\t",
    quotechar='"',
    engine="python",
    dtype=str,
)

# super_domain aus subjects erzeugen
df_all["super_domain"] = df_all["subjects"].apply(map_subjects_to_super)

# Aufräumen
df_all = df_all.dropna(subset=["statement", "label", "super_domain"])
df_all = df_all[df_all["statement"].str.strip() != ""]
df_all = df_all[df_all["label"].isin(["True", "False"])]

print("Super-Domain-Verteilung:")
print(df_all["super_domain"].value_counts())
df_all.to_csv("Results/preprocessed_test_cleaned_binary_with_super_domain_8.csv", sep="\t", index=False)



Super-Domain-Verteilung:
super_domain
economy                473
politics_government    233
law_rights             145
health_social          143
foreign_security       122
society_culture         61
environment_energy      56
misc                    34
Name: count, dtype: int64


In [2]:
from multiagent_pipeline import run_full_pipeline

SUPER_LABELS_8 = [
        "economy",
        "health_social",
        "foreign_security",
        "law_rights",
        "politics_government",
        "environment_energy",
        "society_culture",
        "misc",
]

df_pred = run_full_pipeline(
    base_model_id="meta-llama/Llama-3.1-8B-Instruct",
    router_lora_dir="adapters/llama-liar-statement-domain-lora_8Classes",
    expert_root="adapters/Llama_experts_8Classes",
    super_labels=SUPER_LABELS_8,
    test_path="Results/preprocessed_test_cleaned_binary_with_super_domain_8.csv",
    out_path="Results/Llama_test_predictions_with_experts_8.csv",
    text_col="statement",
    label_col="label",
    domain_col="super_domain",
    sep="\t",
)


Test set size: 1267
True domain distribution:
super_domain
economy                473
politics_government    233
law_rights             145
health_social          143
foreign_security       122
society_culture         61
environment_energy      56
misc                    34
Name: count, dtype: int64

=== Phase 1: Routing (Statement -> super_domain) ===


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🌍 Routing-only: 100%|██████████| 1267/1267 [09:42<00:00,  2.17it/s]



Predicted domain distribution (router):
domain_pred_router
economy                506
politics_government    201
health_social          162
law_rights             139
foreign_security       114
society_culture         65
environment_energy      57
misc                    23
Name: count, dtype: int64


...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: society_culture: 100%|██████████| 65/65 [26:24<00:00, 24.38s/it]


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: misc: 100%|██████████| 23/23 [03:53<00:00, 10.16s/it]


Sample rows with predictions:
                                           statement   super_domain  \
0  Building a wall on the U.S . Mexico border wil...     law_rights   
1  Wisconsin is on pace to double the number of l...        economy   
2  Says John McCain has done nothing to help the ...  health_social   
3  Suzanne Bonamici supports a plan that will cut...  health_social   
4  When asked by a reporter whether hes at the ce...     law_rights   

     domain_pred  label verdict_pred  
0     law_rights   True         True  
1        economy  False         True  
2  health_social  False        False  
3  health_social   True         True  
4        economy  False         True  

🌍 Domain routing accuracy: 0.740
Confusion matrix (rows=true, cols=pred):
[[393  28   2   7  29   3   8   3]
 [ 15 108   3  13   1   0   3   0]
 [  8   3  95   9   2   2   2   1]
 [  7  13   5 106  11   1   2   0]
 [ 55   4   6   3 143  10   8   4]
 [  8   0   1   0   5  39   1   2]
 [  7   3   2   1   6  

In [ ]:
from multiagent_pipeline import run_full_pipeline

SUPER_LABELS_8 = [
        "economy",
        "health_social",
        "foreign_security",
        "law_rights",
        "politics_government",
        "environment_energy",
        "society_culture",
        "misc",
]

df_pred = run_full_pipeline(
    base_model_id="google/gemma-7b-it",
    router_lora_dir="adapters/gemma-liar-statement-domain-lora_8Classes",
    expert_root="adapters/gemma_experts_8Classes",
    super_labels=SUPER_LABELS_8,
    test_path="Results/preprocessed_test_cleaned_binary_with_super_domain_8.csv",
    out_path="Results/gemma_test_predictions_with_experts_8.csv",
    text_col="statement",
    label_col="label",
    domain_col="super_domain",
    sep="\t",
)


In [8]:
evaluate_predictions("Results/gemma_test_predictions_with_experts_8.csv")


✅ Verdict Evaluation (overall)
Samples: 1267
Verdict Accuracy: 0.577

Classification report (True/False):
              precision    recall  f1-score   support

        True       0.61      0.69      0.65       714
       False       0.52      0.43      0.47       553

    accuracy                           0.58      1267
   macro avg       0.56      0.56      0.56      1267
weighted avg       0.57      0.58      0.57      1267

Confusion matrix [rows=true, cols=pred] (True, False):
[[492 222]
 [314 239]]

🌍 Domain Routing Evaluation
Routing samples: 1267
Domain-Routing Accuracy: 0.720
Domain-Routing Confusion Matrix (rows=true, cols=pred):
[[388  31   2  13  17  13   8   1]
 [  8 115   3  14   0   0   3   0]
 [  4   5  92  15   2   2   2   0]
 [  6   7   5 117   7   1   2   0]
 [ 59   9   6  15 111  14  17   2]
 [  8   1   0   0   2  42   3   0]
 [  7   3   3   5   3   2  38   0]
 [ 15   1   2   1   2   2   2   9]]
Domain label order: ['economy', 'health_social', 'foreign_security', 

In [1]:
from multiagent_pipeline import run_full_pipeline

SUPER_LABELS_8 = [
        "economy",
        "health_social",
        "foreign_security",
        "law_rights",
        "politics_government",
        "environment_energy",
        "society_culture",
        "misc",
]

df_pred = run_full_pipeline(
    base_model_id="deepseek-ai/deepseek-llm-7b-base",
    router_lora_dir="adapters/deepseek-liar-statement-domain-lora_8Classes",
    expert_root="adapters/deepseek_experts_8Classes",
    super_labels=SUPER_LABELS_8,
    test_path="Results/preprocessed_test_cleaned_binary_with_super_domain_8.csv",
    out_path="Results/deepseek_test_predictions_with_experts_8.csv",
    text_col="statement",
    label_col="label",
    domain_col="super_domain",
    sep="\t",
)


Test set size: 1267
True domain distribution:
super_domain
economy                473
politics_government    233
law_rights             145
health_social          143
foreign_security       122
society_culture         61
environment_energy      56
misc                    34
Name: count, dtype: int64

=== Phase 1: Routing (Statement -> super_domain) ===


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🌍 Routing-only: 100%|██████████| 1267/1267 [09:18<00:00,  2.27it/s]



Predicted domain distribution (router):
domain_pred_router
economy                482
politics_government    225
health_social          163
law_rights             146
foreign_security       120
society_culture         59
environment_energy      53
misc                    19
Name: count, dtype: int64


...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: society_culture: 100%|██████████| 59/59 [07:10<00:00,  7.30s/it]


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: misc: 100%|██████████| 19/19 [03:57<00:00, 12.48s/it]


Sample rows with predictions:
                                           statement   super_domain  \
0  Building a wall on the U.S . Mexico border wil...     law_rights   
1  Wisconsin is on pace to double the number of l...        economy   
2  Says John McCain has done nothing to help the ...  health_social   
3  Suzanne Bonamici supports a plan that will cut...  health_social   
4  When asked by a reporter whether hes at the ce...     law_rights   

           domain_pred  label verdict_pred  
0           law_rights   True         True  
1              economy  False         True  
2        health_social  False         True  
3        health_social   True         True  
4  politics_government  False         True  

🌍 Domain routing accuracy: 0.744
Confusion matrix (rows=true, cols=pred):
[[388  27   1   9  30   6   8   4]
 [  8 118   3  10   2   0   2   0]
 [  6   3  94  10   5   0   4   0]
 [  2   9  10 107  14   0   3   0]
 [ 43   3   8   6 155  10   7   1]
 [ 11   0   1   0   4 

## 6 Superlabels

In [1]:
domain_to_super_5 = {
    # ----- 1. socioeconomic_policy -----
    "Alcohol": "socioeconomic_policy",
    "agriculture": "socioeconomic_policy",
    "bankruptcy": "socioeconomic_policy",
    "corporations": "socioeconomic_policy",
    "consumer-safety": "socioeconomic_policy",
    "debt": "socioeconomic_policy",
    "deficit": "socioeconomic_policy",
    "economy": "socioeconomic_policy",
    "financial-regulation": "socioeconomic_policy",
    "gas-prices": "socioeconomic_policy",
    "gambling": "socioeconomic_policy",
    "housing": "socioeconomic_policy",
    "income": "socioeconomic_policy",
    "infrastructure": "socioeconomic_policy",
    "job-accomplishments": "socioeconomic_policy",
    "jobs": "socioeconomic_policy",
    "labor": "socioeconomic_policy",
    "market-regulation": "socioeconomic_policy",
    "medicaid": "socioeconomic_policy",
    "pensions": "socioeconomic_policy",
    "poverty": "socioeconomic_policy",
    "retirement": "socioeconomic_policy",
    "small-business": "socioeconomic_policy",
    "social-security": "socioeconomic_policy",
    "stimulus": "socioeconomic_policy",
    "taxes": "socioeconomic_policy",
    "tourism": "socioeconomic_policy",
    "trade": "socioeconomic_policy",
    "unions": "socioeconomic_policy",
    "wealth": "socioeconomic_policy",
    "workers": "socioeconomic_policy",
    "food": "socioeconomic_policy",

    # health_social domains → socioeconomic_policy
    "health-care": "socioeconomic_policy",
    "medicare": "socioeconomic_policy",
    "public-health": "socioeconomic_policy",
    "food-safety": "socioeconomic_policy",
    "drugs": "socioeconomic_policy",
    "marijuana": "socioeconomic_policy",
    "ebola": "socioeconomic_policy",
    "hunger": "socioeconomic_policy",
    "disability": "socioeconomic_policy",
    "women": "socioeconomic_policy",
    "families": "socioeconomic_policy",
    "sexuality": "socioeconomic_policy",
    "veterans": "socioeconomic_policy",
    "children": "socioeconomic_policy",
    "autism": "socioeconomic_policy",
    "homeless": "socioeconomic_policy",

    # ----- 2. foreign_security -----
    "afghanistan": "foreign_security",
    "foreign-policy": "foreign_security",
    "iraq": "foreign_security",
    "israel": "foreign_security",
    "china": "foreign_security",
    "military": "foreign_security",
    "homeland-security": "foreign_security",
    "terrorism": "foreign_security",
    "nuclear": "foreign_security",
    "patriotism": "foreign_security",
    "natural-disasters": "foreign_security",
    "public-safety": "foreign_security",

    # ----- 3. governance_law -----
    "abortion": "governance_law",
    "civil-rights": "governance_law",
    "crime": "governance_law",
    "criminal-justice": "governance_law",
    "guns": "governance_law",
    "legal-issues": "governance_law",
    "privacy": "governance_law",
    "human-rights": "governance_law",
    "supreme-court": "governance_law",
    "kagan-nomination": "governance_law",
    "sotomayor-nomination": "governance_law",
    "islam": "governance_law",
    "immigration": "governance_law",

    "bipartisanship": "governance_law",
    "bush-administration": "governance_law",
    "campaign-advertising": "governance_law",
    "campaign-finance": "governance_law",
    "candidates-biography": "governance_law",
    "congress": "governance_law",
    "congressional-rules": "governance_law",
    "corrections-and-updates": "governance_law",
    "county-government": "governance_law",
    "county-budget": "governance_law",
    "city-government": "governance_law",
    "city-budget": "governance_law",
    "debates": "governance_law",
    "elections": "governance_law",
    "ethics": "governance_law",
    "fake-news": "governance_law",
    "government-efficiency": "governance_law",
    "government-regulation": "governance_law",
    "message-machine": "governance_law",
    "message-machine-2012": "governance_law",
    "message-machine-2014": "governance_law",
    "new-hampshire-2012": "governance_law",
    "occupy-wall-street": "governance_law",
    "politifacts-top-promises": "governance_law",
    "polls": "governance_law",
    "public-service": "governance_law",
    "redistricting": "governance_law",
    "transparency": "governance_law",
    "states": "governance_law",
    "state-budget": "governance_law",
    "state-finances": "governance_law",
    "voting-record": "governance_law",

    # ----- 4. environment_science -----
    "cap-and-trade": "environment_science",
    "climate-change": "environment_science",
    "environment": "environment_science",
    "energy": "environment_science",
    "oil-spill": "environment_science",
    "water": "environment_science",
    "weather": "environment_science",
    "transportation": "environment_science",
    "urban": "environment_science",
    "recreation": "environment_science",

    # frühere misc → environment_science
    "space": "environment_science",
    "technology": "environment_science",

    # ----- 5. society_culture -----
    "education": "society_culture",
    "gays-and-lesbians": "society_culture",
    "diversity": "society_culture",
    "religion": "society_culture",
    "marriage": "society_culture",
    "population": "society_culture",
    "pop-culture": "society_culture",
    "sports": "society_culture",
    "nightlife": "society_culture",
    "history": "society_culture",

    # ----- misc bleibt misc (Medien & irrelevante Kategorien) -----
    "abc-news-week": "misc",
    "pundits": "misc",
    "obama-birth-certificate": "misc",
    "10-news-tampa-bay": "misc",
    "after-the-fact": "misc",
    "lottery": "misc",
}


### Train Domain Routing Model

In [13]:
from train_domain_expert_domain_config import DomainConfig
from train_domain_expert_domain_classifier import train_domain_classifier, evaluate_on_testset


domain_config = DomainConfig(
    domain_to_super=domain_to_super_5,
    # Falls du die Reihenfolge explizit festlegen willst:
    super_labels=[
        "socioeconomic_policy",
        "foreign_security",
        "governance_law",
        "environment_science",
        "society_culture",
        "misc",
    ]
)

# ========= 2) Pfade & Modell-ID =========

base_model_id = "meta-llama/Llama-3.1-8B-Instruct"
train_path = "../2_Finetuning_Masterthesis/LIAR/preprocessed_train_cleaned_binary.csv"
test_path = "../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"
output_dir = "adapters/llama-liar-statement-domain-lora_5Classes"

# ========= 3) Training starten =========

if __name__ == "__main__":
    # Train
    train_domain_classifier(
        train_path=train_path,
        domain_config=domain_config,
        base_model_id=base_model_id,
        output_dir=output_dir,
        num_train_epochs=3,
        learning_rate=5e-4,
        per_device_train_batch_size=4,
        max_length=256,
    )

    # Eval
    evaluate_on_testset(
        test_path=test_path,
        domain_config=domain_config,
        base_model_id=base_model_id,
        lora_dir=output_dir,
    )


Super label distribution (train):
super_domain
socioeconomic_policy    5211
governance_law          2974
foreign_security         971
society_culture          467
environment_science      428
misc                     189
Name: count, dtype: int64


Map:   0%|          | 0/10240 [00:00<?, ? examples/s]

Example tokenized sample keys: dict_keys(['input_ids', 'attention_mask'])
Input length: 151


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


...

Step,Training Loss
50,0.689100
100,0.420600
150,0.410400
200,0.402800
250,0.402200
300,0.407100
350,0.401100
400,0.406700
450,0.408700
500,0.421100


✅ LoRA adapter & tokenizer saved to: adapters/llama-liar-statement-domain-lora_5Classes
Test set size: 1267
True label distribution (test):
super_domain_true
socioeconomic_policy    616
governance_law          378
foreign_security        122
society_culture          61
environment_science      57
misc                     33
Name: count, dtype: int64


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🧠 Routing statements: 100%|██████████| 1267/1267 [09:52<00:00,  2.14it/s]

Predicted label distribution (test):
super_domain_pred
socioeconomic_policy    638
governance_law          373
foreign_security        111
society_culture          76
environment_science      55
misc                     14
Name: count, dtype: int64

Accuracy (super_domain) on test set: 0.796

Classification report (per superlabel):
                      precision    recall  f1-score   support

socioeconomic_policy       0.85      0.88      0.86       616
    foreign_security       0.84      0.76      0.80       122
      governance_law       0.76      0.75      0.76       378
 environment_science       0.67      0.65      0.66        57
     society_culture       0.57      0.70      0.63        61
                misc       0.64      0.27      0.38        33

            accuracy                           0.80      1267
           macro avg       0.72      0.67      0.68      1267
        weighted avg       0.80      0.80      0.79      1267

Confusion matrix (rows=true, cols=pred):
La

In [14]:
from train_domain_expert_domain_config import DomainConfig
from train_domain_expert_domain_classifier import train_domain_classifier, evaluate_on_testset


domain_config = DomainConfig(
    domain_to_super=domain_to_super_5,
    # Falls du die Reihenfolge explizit festlegen willst:
    super_labels=[
        "socioeconomic_policy",
        "foreign_security",
        "governance_law",
        "environment_science",
        "society_culture",
        "misc",
    ]
)

# ========= 2) Pfade & Modell-ID =========

base_model_id = "google/gemma-7b-it"
train_path = "../2_Finetuning_Masterthesis/LIAR/preprocessed_train_cleaned_binary.csv"
test_path = "../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"
output_dir = "adapters/gemma-liar-statement-domain-lora_5Classes"

# ========= 3) Training starten =========

if __name__ == "__main__":
    # Train
    train_domain_classifier(
        train_path=train_path,
        domain_config=domain_config,
        base_model_id=base_model_id,
        output_dir=output_dir,
        num_train_epochs=3,
        learning_rate=5e-4,
        per_device_train_batch_size=4,
        max_length=256,
    )

    # Eval
    evaluate_on_testset(
        test_path=test_path,
        domain_config=domain_config,
        base_model_id=base_model_id,
        lora_dir=output_dir,
    )


Super label distribution (train):
super_domain
socioeconomic_policy    5211
governance_law          2974
foreign_security         971
society_culture          467
environment_science      428
misc                     189
Name: count, dtype: int64


Map:   0%|          | 0/10240 [00:00<?, ? examples/s]

Example tokenized sample keys: dict_keys(['input_ids', 'attention_mask'])
Input length: 133


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


...

Step,Training Loss
50,1.593100
100,0.709200
150,0.623700
200,0.564400
250,0.580600
300,0.577800
350,0.581100
400,0.589200
450,0.592200
500,0.574000


✅ LoRA adapter & tokenizer saved to: adapters/gemma-liar-statement-domain-lora_5Classes
Test set size: 1267
True label distribution (test):
super_domain_true
socioeconomic_policy    616
governance_law          378
foreign_security        122
society_culture          61
environment_science      57
misc                     33
Name: count, dtype: int64


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🧠 Routing statements: 100%|██████████| 1267/1267 [04:27<00:00,  4.74it/s]

Predicted label distribution (test):
super_domain_pred
socioeconomic_policy    589
governance_law          387
foreign_security        121
society_culture          80
environment_science      80
misc                     10
Name: count, dtype: int64

Accuracy (super_domain) on test set: 0.770

Classification report (per superlabel):
                      precision    recall  f1-score   support

socioeconomic_policy       0.87      0.84      0.85       616
    foreign_security       0.81      0.80      0.81       122
      governance_law       0.72      0.73      0.72       378
 environment_science       0.53      0.74      0.61        57
     society_culture       0.46      0.61      0.52        61
                misc       0.60      0.18      0.28        33

            accuracy                           0.77      1267
           macro avg       0.66      0.65      0.63      1267
        weighted avg       0.78      0.77      0.77      1267

Confusion matrix (rows=true, cols=pred):
La

In [15]:
from train_domain_expert_domain_config import DomainConfig
from train_domain_expert_domain_classifier import train_domain_classifier, evaluate_on_testset


domain_config = DomainConfig(
    domain_to_super=domain_to_super_5,
    # Falls du die Reihenfolge explizit festlegen willst:
    super_labels=[
        "socioeconomic_policy",
        "foreign_security",
        "governance_law",
        "environment_science",
        "society_culture",
        "misc",
    ]
)

# ========= 2) Pfade & Modell-ID =========

base_model_id = "deepseek-ai/deepseek-llm-7b-base"
train_path = "../2_Finetuning_Masterthesis/LIAR/preprocessed_train_cleaned_binary.csv"
test_path = "../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"
output_dir = "adapters/deepseek-liar-statement-domain-lora_5Classes"

# ========= 3) Training starten =========

if __name__ == "__main__":
    # Train
    train_domain_classifier(
        train_path=train_path,
        domain_config=domain_config,
        base_model_id=base_model_id,
        output_dir=output_dir,
        num_train_epochs=3,
        learning_rate=5e-4,
        per_device_train_batch_size=4,
        max_length=256,
    )

    # Eval
    evaluate_on_testset(
        test_path=test_path,
        domain_config=domain_config,
        base_model_id=base_model_id,
        lora_dir=output_dir,
    )


Super label distribution (train):
super_domain
socioeconomic_policy    5211
governance_law          2974
foreign_security         971
society_culture          467
environment_science      428
misc                     189
Name: count, dtype: int64


Map:   0%|          | 0/10240 [00:00<?, ? examples/s]

Example tokenized sample keys: dict_keys(['input_ids', 'attention_mask'])
Input length: 140


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 18,739,200 || all params: 6,929,104,896 || trainable%: 0.2704


...

Step,Training Loss
50,0.623100
100,0.439000
150,0.425800
200,0.410600
250,0.410000
300,0.416300
350,0.406500
400,0.411400
450,0.413900
500,0.425500


✅ LoRA adapter & tokenizer saved to: adapters/deepseek-liar-statement-domain-lora_5Classes
Test set size: 1267
True label distribution (test):
super_domain_true
socioeconomic_policy    616
governance_law          378
foreign_security        122
society_culture          61
environment_science      57
misc                     33
Name: count, dtype: int64


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🧠 Routing statements: 100%|██████████| 1267/1267 [09:08<00:00,  2.31it/s]

Predicted label distribution (test):
super_domain_pred
socioeconomic_policy    632
governance_law          376
foreign_security        126
society_culture          61
environment_science      61
misc                     11
Name: count, dtype: int64

Accuracy (super_domain) on test set: 0.780

Classification report (per superlabel):
                      precision    recall  f1-score   support

socioeconomic_policy       0.84      0.86      0.85       616
    foreign_security       0.79      0.82      0.81       122
      governance_law       0.73      0.73      0.73       378
 environment_science       0.64      0.68      0.66        57
     society_culture       0.56      0.56      0.56        61
                misc       0.82      0.27      0.41        33

            accuracy                           0.78      1267
           macro avg       0.73      0.65      0.67      1267
        weighted avg       0.78      0.78      0.78      1267

Confusion matrix (rows=true, cols=pred):
La

### Train-Experts

In [2]:
import pandas as pd
SUPER_LABELS = [
        "socioeconomic_policy",
        "foreign_security",
        "governance_law",
        "environment_science",
        "society_culture",
        "misc",
    ]
def map_subjects_to_super(subjects_str: str) -> str:
    """
    Map LIAR 'subjects' (comma-separated topics) to exactly one super label.
    """
    if not isinstance(subjects_str, str) or not subjects_str.strip():
        return "misc"

    domains = [s.strip() for s in subjects_str.split(",") if s.strip()]
    super_labels = {domain_to_super_5.get(d, "misc") for d in domains}

    # priorisierte Reihenfolge: erstes vorkommendes Label aus SUPER_LABELS
    for label in SUPER_LABELS:
        if label in super_labels:
            return label
    return "misc"

data_path = "../2_Finetuning_Masterthesis/LIAR/preprocessed_train_cleaned_binary.csv"  # anpassen falls nötig

df_all = pd.read_csv(
    data_path,
    sep="\t",
    quotechar='"',
    engine="python",
    dtype=str,
)

# super_domain aus subjects erzeugen
df_all["super_domain"] = df_all["subjects"].apply(map_subjects_to_super)

# Aufräumen
df_all = df_all.dropna(subset=["statement", "label", "super_domain"])
df_all = df_all[df_all["statement"].str.strip() != ""]
df_all = df_all[df_all["label"].isin(["True", "False"])]

print("Super-Domain-Verteilung:")
print(df_all["super_domain"].value_counts())


Super-Domain-Verteilung:
super_domain
socioeconomic_policy    5211
governance_law          2974
foreign_security         971
society_culture          467
environment_science      428
misc                     189
Name: count, dtype: int64


In [3]:
df_all.to_csv("Results/preprocessed_train_cleaned_binary_with_super_domain_5.csv", sep="\t", index=False)


In [4]:
import pandas as pd

data_path = "Results/preprocessed_train_cleaned_binary_with_super_domain_5.csv"

df_all = pd.read_csv(
    data_path,
    sep="\t",
    quotechar='"',
    engine="python",
    dtype=str,
)

# Sicherstellen, dass wir nur sinnvolle Zeilen haben
df_all = df_all.dropna(subset=["statement", "label", "super_domain"])
df_all = df_all[df_all["statement"].str.strip() != ""]
df_all = df_all[df_all["label"].isin(["True", "False"])]

print("Gesamtverteilung super_domain:")
print(df_all["super_domain"].value_counts())

SUPER_LABELS = [
        "socioeconomic_policy",
        "foreign_security",
        "governance_law",
        "environment_science",
        "society_culture",
        "misc",
]

# Pro Domain ein DataFrame
domain_dfs = {
    domain: df_all[df_all["super_domain"] == domain].reset_index(drop=True)
    for domain in SUPER_LABELS
}

for dom, df_dom in domain_dfs.items():
    print(f"\nDomain: {dom} -> {len(df_dom)} Beispiele")


Gesamtverteilung super_domain:
super_domain
socioeconomic_policy    5211
governance_law          2974
foreign_security         971
society_culture          467
environment_science      428
misc                     189
Name: count, dtype: int64

Domain: socioeconomic_policy -> 5211 Beispiele

Domain: foreign_security -> 971 Beispiele

Domain: governance_law -> 2974 Beispiele

Domain: environment_science -> 428 Beispiele

Domain: society_culture -> 467 Beispiele

Domain: misc -> 189 Beispiele


In [5]:
# run_train_experts.py (oder Notebook-Zelle)
import sys, os
import pandas as pd

# falls du in Jupyter bist:
sys.path.append(os.getcwd())

from train_expert_expert_config import ExpertConfig
from train_expert_expert_trainer import train_all_experts

# 1) Trainingsdaten laden – EIN gemeinsamer DataFrame
train_path = "Results/preprocessed_train_cleaned_binary_with_super_domain_5.csv"

df_train = pd.read_csv(
    train_path,
    sep="\t",
    quotechar='"',
    engine="python",
    dtype=str,
)

# 2) Konfiguration der Experten (Superdomains + Spaltennamen)
config = ExpertConfig(
    super_domains=[
        "socioeconomic_policy",
        "foreign_security",
        "governance_law",
        "environment_science",
        "society_culture",
        "misc",
    ],
    base_model_id="meta-llama/Llama-3.1-8B-Instruct",
    text_column="statement",
    label_column="label",
    domain_column="super_domain",
    max_length=256,
    num_train_epochs=3,
    learning_rate=5e-4,
    per_device_train_batch_size=4,
)

# 3) Output-Root – dort bekommt jede Domain seinen eigenen Unterordner
output_root = "adapters/Llama_experts_5Classes"

saved_paths = train_all_experts(
    df=df_train,
    config=config,
    output_root=output_root,
)

print("\nFertig! Modelle gespeichert unter:")
for dom, p in saved_paths.items():
    print(f"  {dom}: {p}")


2026-02-02 06:19:59.986569: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.



=== Training expert for domain: socioeconomic_policy ===
Samples: 5211


Map:   0%|          | 0/5211 [00:00<?, ? examples/s]

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 20,971,520 || all params: 8,051,232,768 || trainable%: 0.2605


🚀 Training:   0%|          | 0/72 [00:00<?, ?it/s]

Step,Training Loss
25,0.846400
50,0.316900


✔️ Expert-LoRA for 'misc' saved to adapters/Llama_experts_5Classes/misc

Fertig! Modelle gespeichert unter:
  socioeconomic_policy: adapters/Llama_experts_5Classes/socioeconomic_policy
  foreign_security: adapters/Llama_experts_5Classes/foreign_security
  governance_law: adapters/Llama_experts_5Classes/governance_law
  environment_science: adapters/Llama_experts_5Classes/environment_science
  society_culture: adapters/Llama_experts_5Classes/society_culture
  misc: adapters/Llama_experts_5Classes/misc


In [6]:
# run_train_experts.py (oder Notebook-Zelle)
import sys, os
import pandas as pd

# falls du in Jupyter bist:
sys.path.append(os.getcwd())

from train_expert_expert_config import ExpertConfig
from train_expert_expert_trainer import train_all_experts

# 1) Trainingsdaten laden – EIN gemeinsamer DataFrame
train_path = "Results/preprocessed_train_cleaned_binary_with_super_domain_5.csv"

df_train = pd.read_csv(
    train_path,
    sep="\t",
    quotechar='"',
    engine="python",
    dtype=str,
)

# 2) Konfiguration der Experten (Superdomains + Spaltennamen)
config = ExpertConfig(
    super_domains=[
        "socioeconomic_policy",
        "foreign_security",
        "governance_law",
        "environment_science",
        "society_culture",
        "misc",
    ],
    base_model_id="google/gemma-7b-it",
    text_column="statement",
    label_column="label",
    domain_column="super_domain",
    max_length=256,
    num_train_epochs=3,
    learning_rate=5e-4,
    per_device_train_batch_size=4,
)

# 3) Output-Root – dort bekommt jede Domain seinen eigenen Unterordner
output_root = "adapters/gemma_experts_5Classes"

saved_paths = train_all_experts(
    df=df_train,
    config=config,
    output_root=output_root,
)

print("\nFertig! Modelle gespeichert unter:")
for dom, p in saved_paths.items():
    print(f"  {dom}: {p}")



=== Training expert for domain: socioeconomic_policy ===
Samples: 5211


Map:   0%|          | 0/5211 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 25,001,984 || all params: 8,562,682,880 || trainable%: 0.2920


🚀 Training:   0%|          | 0/1956 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 25,001,984 || all params: 8,562,682,880 || trainable%: 0.2920


🚀 Training:   0%|          | 0/72 [00:00<?, ?it/s]

Step,Training Loss
25,2.441500
50,0.579900


✔️ Expert-LoRA for 'misc' saved to adapters/gemma_experts_5Classes/misc

Fertig! Modelle gespeichert unter:
  socioeconomic_policy: adapters/gemma_experts_5Classes/socioeconomic_policy
  foreign_security: adapters/gemma_experts_5Classes/foreign_security
  governance_law: adapters/gemma_experts_5Classes/governance_law
  environment_science: adapters/gemma_experts_5Classes/environment_science
  society_culture: adapters/gemma_experts_5Classes/society_culture
  misc: adapters/gemma_experts_5Classes/misc


In [7]:
# run_train_experts.py (oder Notebook-Zelle)
import sys, os
import pandas as pd

# falls du in Jupyter bist:
sys.path.append(os.getcwd())

from train_expert_expert_config import ExpertConfig
from train_expert_expert_trainer import train_all_experts

# 1) Trainingsdaten laden – EIN gemeinsamer DataFrame
train_path = "Results/preprocessed_train_cleaned_binary_with_super_domain_5.csv"

df_train = pd.read_csv(
    train_path,
    sep="\t",
    quotechar='"',
    engine="python",
    dtype=str,
)

# 2) Konfiguration der Experten (Superdomains + Spaltennamen)
config = ExpertConfig(
    super_domains=[
        "socioeconomic_policy",
        "foreign_security",
        "governance_law",
        "environment_science",
        "society_culture",
        "misc",
    ],
    base_model_id="deepseek-ai/deepseek-llm-7b-base",
    text_column="statement",
    label_column="label",
    domain_column="super_domain",
    max_length=256,
    num_train_epochs=3,
    learning_rate=5e-4,
    per_device_train_batch_size=4,
)

# 3) Output-Root – dort bekommt jede Domain seinen eigenen Unterordner
output_root = "adapters/deepseek_experts_5Classes"

saved_paths = train_all_experts(
    df=df_train,
    config=config,
    output_root=output_root,
)

print("\nFertig! Modelle gespeichert unter:")
for dom, p in saved_paths.items():
    print(f"  {dom}: {p}")



=== Training expert for domain: socioeconomic_policy ===
Samples: 5211


Map:   0%|          | 0/5211 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 18,739,200 || all params: 6,929,104,896 || trainable%: 0.2704


🚀 Training:   0%|          | 0/1956 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 18,739,200 || all params: 6,929,104,896 || trainable%: 0.2704


🚀 Training:   0%|          | 0/72 [00:00<?, ?it/s]

Step,Training Loss
25,0.737000
50,0.385500


✔️ Expert-LoRA for 'misc' saved to adapters/deepseek_experts_5Classes/misc

Fertig! Modelle gespeichert unter:
  socioeconomic_policy: adapters/deepseek_experts_5Classes/socioeconomic_policy
  foreign_security: adapters/deepseek_experts_5Classes/foreign_security
  governance_law: adapters/deepseek_experts_5Classes/governance_law
  environment_science: adapters/deepseek_experts_5Classes/environment_science
  society_culture: adapters/deepseek_experts_5Classes/society_culture
  misc: adapters/deepseek_experts_5Classes/misc


### Evaluation

In [2]:
import pandas as pd
SUPER_LABELS = [
        "socioeconomic_policy",
        "foreign_security",
        "governance_law",
        "environment_science",
        "society_culture",
        "misc",
]
def map_subjects_to_super(subjects_str: str) -> str:
    """
    Map LIAR 'subjects' (comma-separated topics) to exactly one super label.
    """
    if not isinstance(subjects_str, str) or not subjects_str.strip():
        return "misc"

    domains = [s.strip() for s in subjects_str.split(",") if s.strip()]
    super_labels = {domain_to_super_5.get(d, "misc") for d in domains}

    # priorisierte Reihenfolge: erstes vorkommendes Label aus SUPER_LABELS
    for label in SUPER_LABELS:
        if label in super_labels:
            return label
    return "misc"

data_path = "../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"  # anpassen falls nötig

df_all = pd.read_csv(
    data_path,
    sep="\t",
    quotechar='"',
    engine="python",
    dtype=str,
)

# super_domain aus subjects erzeugen
df_all["super_domain"] = df_all["subjects"].apply(map_subjects_to_super)

# Aufräumen
df_all = df_all.dropna(subset=["statement", "label", "super_domain"])
df_all = df_all[df_all["statement"].str.strip() != ""]
df_all = df_all[df_all["label"].isin(["True", "False"])]

print("Super-Domain-Verteilung:")
print(df_all["super_domain"].value_counts())
df_all.to_csv("Results/preprocessed_test_cleaned_binary_with_super_domain_5.csv", sep="\t", index=False)



Super-Domain-Verteilung:
super_domain
socioeconomic_policy    616
governance_law          378
foreign_security        122
society_culture          61
environment_science      57
misc                     33
Name: count, dtype: int64


In [ ]:
from multiagent_pipeline import run_full_pipeline

SUPER_LABELS_5 = [
        "socioeconomic_policy",
        "foreign_security",
        "governance_law",
        "environment_science",
        "society_culture",
        "misc",
]

df_pred = run_full_pipeline(
    base_model_id="meta-llama/Llama-3.1-8B-Instruct",
    router_lora_dir="adapters/llama-liar-statement-domain-lora_5Classes",
    expert_root="adapters/Llama_experts_5Classes",
    super_labels=SUPER_LABELS_5,
    test_path="Results/preprocessed_test_cleaned_binary_with_super_domain_5.csv",
    out_path="Results/Llama_test_predictions_with_experts_5.csv",
    text_col="statement",
    label_col="label",
    domain_col="super_domain",
    sep="\t",
)


Test set size: 1267
True domain distribution:
super_domain
socioeconomic_policy    616
governance_law          378
foreign_security        122
society_culture          61
environment_science      57
misc                     33
Name: count, dtype: int64

=== Phase 1: Routing (Statement -> super_domain) ===


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🌍 Routing-only: 100%|██████████| 1267/1267 [10:13<00:00,  2.07it/s]



Predicted domain distribution (router):
domain_pred_router
socioeconomic_policy    699
governance_law          330
foreign_security        106
society_culture          64
environment_science      54
misc                     14
Name: count, dtype: int64


...

Device set to use cuda:0
🧠 Expert: socioeconomic_policy: 100%|██████████| 699/699 [5:34:59<00:00, 28.75s/it]  


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: foreign_security: 100%|██████████| 106/106 [19:10<00:00, 10.85s/it]


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: governance_law:   1%|          | 4/330 [01:22<1:58:04, 21.73s/it]

In [ ]:
from multiagent_pipeline import run_full_pipeline

SUPER_LABELS_5 = [
        "socioeconomic_policy",
        "foreign_security",
        "governance_law",
        "environment_science",
        "society_culture",
        "misc",
]

df_pred = run_full_pipeline(
    base_model_id="google/gemma-7b-it",
    router_lora_dir="adapters/gemma-liar-statement-domain-lora_5Classes",
    expert_root="adapters/gemma_experts_5Classes",
    super_labels=SUPER_LABELS_5,
    test_path="Results/preprocessed_test_cleaned_binary_with_super_domain_5.csv",
    out_path="Results/gemma_test_predictions_with_experts_5.csv",
    text_col="statement",
    label_col="label",
    domain_col="super_domain",
    sep="\t",
)


Test set size: 1267
True domain distribution:
super_domain
socioeconomic_policy    616
governance_law          378
foreign_security        122
society_culture          61
environment_science      57
misc                     33
Name: count, dtype: int64

=== Phase 1: Routing (Statement -> super_domain) ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🌍 Routing-only: 100%|██████████| 1267/1267 [05:31<00:00,  3.82it/s]



Predicted domain distribution (router):
domain_pred_router
socioeconomic_policy    594
governance_law          376
foreign_security        119
society_culture          83
environment_science      83
misc                     12
Name: count, dtype: int64


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
🧠 Expert: socioeconomic_policy:   0%|          | 1/594 [00:48<7:56:51, 48.25s/it]

In [6]:
from multiagent_pipeline import run_full_pipeline

SUPER_LABELS_5 = [
        "socioeconomic_policy",
        "foreign_security",
        "governance_law",
        "environment_science",
        "society_culture",
        "misc",
]

df_pred = run_full_pipeline(
    base_model_id="deepseek-ai/deepseek-llm-7b-base",
    router_lora_dir="adapters/deepseek-liar-statement-domain-lora_5Classes",
    expert_root="adapters/deepseek_experts_5Classes",
    super_labels=SUPER_LABELS_5,
    test_path="Results/preprocessed_test_cleaned_binary_with_super_domain_5.csv",
    out_path="Results/deepseek_test_predictions_with_experts_5.csv",
    text_col="statement",
    label_col="label",
    domain_col="super_domain",
    sep="\t",
)


Test set size: 1267
True domain distribution:
super_domain
socioeconomic_policy    616
governance_law          378
foreign_security        122
society_culture          61
environment_science      57
misc                     33
Name: count, dtype: int64

=== Phase 1: Routing (Statement -> super_domain) ===


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🌍 Routing-only: 100%|██████████| 1267/1267 [09:52<00:00,  2.14it/s]



Predicted domain distribution (router):
domain_pred_router
socioeconomic_policy    640
governance_law          394
foreign_security        117
environment_science      58
society_culture          49
misc                      9
Name: count, dtype: int64


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: society_culture: 100%|██████████| 49/49 [09:23<00:00, 11.51s/it]


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: misc: 100%|██████████| 9/9 [00:58<00:00,  6.53s/it]


Sample rows with predictions:
                                           statement          super_domain  \
0  Building a wall on the U.S . Mexico border wil...        governance_law   
1  Wisconsin is on pace to double the number of l...  socioeconomic_policy   
2  Says John McCain has done nothing to help the ...  socioeconomic_policy   
3  Suzanne Bonamici supports a plan that will cut...  socioeconomic_policy   
4  When asked by a reporter whether hes at the ce...        governance_law   

            domain_pred  label verdict_pred  
0        governance_law   True         True  
1  socioeconomic_policy  False         True  
2  socioeconomic_policy  False        False  
3  socioeconomic_policy   True        False  
4        governance_law  False        False  

🌍 Domain routing accuracy: 0.788
Confusion matrix (rows=true, cols=pred):
[[538   2  63   7   6   0]
 [  8  99  12   0   3   0]
 [ 54  11 290   9  10   4]
 [  8   1   7  39   2   0]
 [  9   4  18   1  28   1]
 [ 23   0   4 

## 13 Superlabels

In [8]:
domain_to_super_12 = {

    # ----- 1. macro_econ -----
    "economy": "macro_econ",
    "deficit": "macro_econ",
    "debt": "macro_econ",
    "stimulus": "macro_econ",
    "taxes": "macro_econ",
    "trade": "macro_econ",

    # ----- 2. jobs_labor -----
    "jobs": "jobs_labor",
    "job-accomplishments": "jobs_labor",
    "labor": "jobs_labor",
    "unions": "jobs_labor",
    "workers": "jobs_labor",
    "income": "jobs_labor",

    # ----- 3. business_finance -----
    "corporations": "business_finance",
    "bankruptcy": "business_finance",
    "financial-regulation": "business_finance",
    "market-regulation": "business_finance",
    "small-business": "business_finance",

    # ----- 4. cost_of_living_housing -----
    "housing": "cost_of_living_housing",
    "gas-prices": "cost_of_living_housing",
    "poverty": "cost_of_living_housing",
    "wealth": "cost_of_living_housing",
    "retirement": "cost_of_living_housing",
    "pensions": "cost_of_living_housing",
    "social-security": "cost_of_living_housing",
    "tourism": "cost_of_living_housing",

    # ----- 5. healthcare_policy -----
    "health-care": "healthcare_policy",
    "medicare": "healthcare_policy",
    "medicaid": "healthcare_policy",
    "public-health": "healthcare_policy",

    # ----- 6. public_health_crises -----
    "ebola": "public_health_crises",
    "natural-disasters": "public_health_crises",
    "public-safety": "public_health_crises",
    "food-safety": "public_health_crises",
    "consumer-safety": "public_health_crises",

    # ----- 7. substances_gambling -----
    "drugs": "substances_gambling",
    "marijuana": "substances_gambling",
    "Alcohol": "substances_gambling",
    "gambling": "substances_gambling",
    "lottery": "substances_gambling",

    # ----- 8. family_demographics -----
    "women": "family_demographics",
    "families": "family_demographics",
    "children": "family_demographics",
    "veterans": "family_demographics",
    "disability": "family_demographics",
    "autism": "family_demographics",
    "homeless": "family_demographics",
    "hunger": "family_demographics",
    "population": "family_demographics",
    "food": "family_demographics",

    # ----- 9. law_crime_rights -----
    "crime": "law_crime_rights",
    "criminal-justice": "law_crime_rights",
    "guns": "law_crime_rights",
    "abortion": "law_crime_rights",
    "civil-rights": "law_crime_rights",
    "human-rights": "law_crime_rights",
    "privacy": "law_crime_rights",
    "immigration": "law_crime_rights",
    "islam": "law_crime_rights",

    # ----- 10. institutions_elections -----
    "congress": "institutions_elections",
    "congressional-rules": "institutions_elections",
    "elections": "institutions_elections",
    "polls": "institutions_elections",
    "debates": "institutions_elections",
    "campaign-finance": "institutions_elections",
    "campaign-advertising": "institutions_elections",
    "redistricting": "institutions_elections",
    "voting-record": "institutions_elections",
    "candidates-biography": "institutions_elections",
    "ethics": "institutions_elections",
    "transparency": "institutions_elections",
    "bipartisanship": "institutions_elections",
    "government-regulation": "institutions_elections",
    "government-efficiency": "institutions_elections",
    "public-service": "institutions_elections",
    "states": "institutions_elections",
    "state-budget": "institutions_elections",
    "state-finances": "institutions_elections",
    "county-government": "institutions_elections",
    "county-budget": "institutions_elections",
    "city-government": "institutions_elections",
    "city-budget": "institutions_elections",
    "bush-administration": "institutions_elections",

    # ----- 11. foreign_affairs_security -----
    "foreign-policy": "foreign_affairs_security",
    "afghanistan": "foreign_affairs_security",
    "iraq": "foreign_affairs_security",
    "israel": "foreign_affairs_security",
    "china": "foreign_affairs_security",
    "military": "foreign_affairs_security",
    "terrorism": "foreign_affairs_security",
    "homeland-security": "foreign_affairs_security",
    "nuclear": "foreign_affairs_security",
    "patriotism": "foreign_affairs_security",

    # ----- 12. environment_energy_infra -----
    "climate-change": "environment_energy_infra",
    "cap-and-trade": "environment_energy_infra",
    "environment": "environment_energy_infra",
    "energy": "environment_energy_infra",
    "oil-spill": "environment_energy_infra",
    "water": "environment_energy_infra",
    "weather": "environment_energy_infra",
    "transportation": "environment_energy_infra",
    "urban": "environment_energy_infra",
    "infrastructure": "environment_energy_infra",
    "recreation": "environment_energy_infra",

    # ----- media / meta (optional separate handling) -----
    "abc-news-week": "media_meta",
    "pundits": "media_meta",
    "after-the-fact": "media_meta",
    "corrections-and-updates": "media_meta",
    "fake-news": "media_meta",
    "message-machine": "media_meta",
    "message-machine-2012": "media_meta",
    "message-machine-2014": "media_meta",
    "new-hampshire-2012": "media_meta",
    "occupy-wall-street": "media_meta",
    "politifacts-top-promises": "media_meta",
    "10-news-tampa-bay": "media_meta",
    "obama-birth-certificate": "media_meta",
    "technology": "media_meta",
    "space": "media_meta",
    "history": "media_meta",
    "education": "media_meta",
    "religion": "media_meta",
    "marriage": "media_meta",
    "gays-and-lesbians": "media_meta",
    "diversity": "media_meta",
    "pop-culture": "media_meta",
    "sports": "media_meta",
    "nightlife": "media_meta",
}


### Train Domain Routing Model

In [12]:
from train_domain_expert_domain_config import DomainConfig
from train_domain_expert_domain_classifier import train_domain_classifier, evaluate_on_testset


domain_config = DomainConfig(
    domain_to_super=domain_to_super_12,
    # Falls du die Reihenfolge explizit festlegen willst:
    super_labels=[
        "macro_econ",
        "jobs_labor",
        "business_finance",
        "cost_of_living_housing",
        "healthcare_policy",
        "public_health_crises",
        "substances_gambling",
        "family_demographics",
        "law_crime_rights",
        "institutions_elections",
        "foreign_affairs_security",
        "environment_energy_infra",
        "media_meta",
    ]
)

# ========= 2) Pfade & Modell-ID =========

base_model_id = "meta-llama/Llama-3.1-8B-Instruct"
train_path = "../2_Finetuning_Masterthesis/LIAR/preprocessed_train_cleaned_binary.csv"
test_path = "../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"
output_dir = "adapters/llama-liar-statement-domain-lora_12Classes"

# ========= 3) Training starten =========

if __name__ == "__main__":
    # Train
    train_domain_classifier(
        train_path=train_path,
        domain_config=domain_config,
        base_model_id=base_model_id,
        output_dir=output_dir,
        num_train_epochs=3,
        learning_rate=5e-4,
        per_device_train_batch_size=4,
        max_length=256,
    )

    # Eval
    evaluate_on_testset(
        test_path=test_path,
        domain_config=domain_config,
        base_model_id=base_model_id,
        lora_dir=output_dir,
    )


Super label distribution (train):
super_domain
macro_econ                  2606
institutions_elections      1798
law_crime_rights            1245
healthcare_policy           1087
jobs_labor                   764
foreign_affairs_security     651
media_meta                   571
environment_energy_infra     468
family_demographics          317
cost_of_living_housing       295
business_finance             189
public_health_crises         139
substances_gambling          110
Name: count, dtype: int64


Map:   0%|          | 0/10240 [00:00<?, ? examples/s]

Example tokenized sample keys: dict_keys(['input_ids', 'attention_mask'])
Input length: 197


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 20,971,520 || all params: 8,051,232,768 || trainable%: 0.2605


...

Step,Training Loss
50,0.602300
100,0.324600
150,0.314900
200,0.309400
250,0.308900
300,0.313300
350,0.308700
400,0.313600
450,0.315300
500,0.323300


✅ LoRA adapter & tokenizer saved to: adapters/llama-liar-statement-domain-lora_12Classes
Test set size: 1267
True label distribution (test):
super_domain_true
macro_econ                  316
institutions_elections      256
law_crime_rights            131
healthcare_policy           114
jobs_labor                  107
foreign_affairs_security     81
media_meta                   79
environment_energy_infra     61
cost_of_living_housing       44
family_demographics          28
business_finance             23
public_health_crises         14
substances_gambling          13
Name: count, dtype: int64


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🧠 Routing statements: 100%|██████████| 1267/1267 [09:59<00:00,  2.11it/s]

Predicted label distribution (test):
super_domain_pred
macro_econ                  368
institutions_elections      223
law_crime_rights            145
healthcare_policy           130
foreign_affairs_security     88
media_meta                   85
environment_energy_infra     66
jobs_labor                   59
family_demographics          37
cost_of_living_housing       22
business_finance             20
substances_gambling          16
public_health_crises          8
Name: count, dtype: int64

Accuracy (super_domain) on test set: 0.677

Classification report (per superlabel):
                          precision    recall  f1-score   support

              macro_econ       0.71      0.83      0.77       316
              jobs_labor       0.61      0.34      0.43       107
        business_finance       0.35      0.30      0.33        23
  cost_of_living_housing       0.64      0.32      0.42        44
       healthcare_policy       0.75      0.86      0.80       114
    public_health_cri

In [14]:
from train_domain_expert_domain_config import DomainConfig
from train_domain_expert_domain_classifier import train_domain_classifier, evaluate_on_testset


domain_config = DomainConfig(
    domain_to_super=domain_to_super_12,
    # Falls du die Reihenfolge explizit festlegen willst:
    super_labels=[
        "macro_econ",
        "jobs_labor",
        "business_finance",
        "cost_of_living_housing",
        "healthcare_policy",
        "public_health_crises",
        "substances_gambling",
        "family_demographics",
        "law_crime_rights",
        "institutions_elections",
        "foreign_affairs_security",
        "environment_energy_infra",
        "media_meta",
    ]
)

# ========= 2) Pfade & Modell-ID =========

base_model_id = "google/gemma-7b-it"
train_path = "../2_Finetuning_Masterthesis/LIAR/preprocessed_train_cleaned_binary.csv"
test_path = "../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"
output_dir = "adapters/gemma-liar-statement-domain-lora_12Classes"

# ========= 3) Training starten =========

if __name__ == "__main__":
    # Train
    train_domain_classifier(
        train_path=train_path,
        domain_config=domain_config,
        base_model_id=base_model_id,
        output_dir=output_dir,
        num_train_epochs=3,
        learning_rate=5e-4,
        per_device_train_batch_size=4,
        max_length=256,
    )

    # Eval
    evaluate_on_testset(
        test_path=test_path,
        domain_config=domain_config,
        base_model_id=base_model_id,
        lora_dir=output_dir,
    )


Super label distribution (train):
super_domain
macro_econ                  2606
institutions_elections      1798
law_crime_rights            1245
healthcare_policy           1087
jobs_labor                   764
foreign_affairs_security     651
media_meta                   571
environment_energy_infra     468
family_demographics          317
cost_of_living_housing       295
business_finance             189
public_health_crises         139
substances_gambling          110
Name: count, dtype: int64


Map:   0%|          | 0/10240 [00:00<?, ? examples/s]

Example tokenized sample keys: dict_keys(['input_ids', 'attention_mask'])
Input length: 188


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


...

Step,Training Loss
50,1.162000
100,0.538100
150,0.460700
200,0.419500
250,0.392300
300,0.386300
350,0.375200
400,0.379400
450,0.377800
500,0.392400


✅ LoRA adapter & tokenizer saved to: adapters/gemma-liar-statement-domain-lora_12Classes
Test set size: 1267
True label distribution (test):
super_domain_true
macro_econ                  316
institutions_elections      256
law_crime_rights            131
healthcare_policy           114
jobs_labor                  107
foreign_affairs_security     81
media_meta                   79
environment_energy_infra     61
cost_of_living_housing       44
family_demographics          28
business_finance             23
public_health_crises         14
substances_gambling          13
Name: count, dtype: int64


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🧠 Routing statements: 100%|██████████| 1267/1267 [05:32<00:00,  3.81it/s]

Predicted label distribution (test):
super_domain_pred
macro_econ                  272
institutions_elections      198
law_crime_rights            148
healthcare_policy           123
jobs_labor                  120
media_meta                  104
foreign_affairs_security     79
environment_energy_infra     72
cost_of_living_housing       54
family_demographics          42
business_finance             30
public_health_crises         17
substances_gambling           8
Name: count, dtype: int64

Accuracy (super_domain) on test set: 0.658

Classification report (per superlabel):
                          precision    recall  f1-score   support

              macro_econ       0.79      0.68      0.73       316
              jobs_labor       0.50      0.56      0.53       107
        business_finance       0.30      0.39      0.34        23
  cost_of_living_housing       0.48      0.59      0.53        44
       healthcare_policy       0.80      0.86      0.83       114
    public_health_cri

In [15]:
from train_domain_expert_domain_config import DomainConfig
from train_domain_expert_domain_classifier import train_domain_classifier, evaluate_on_testset


domain_config = DomainConfig(
    domain_to_super=domain_to_super_12,
    # Falls du die Reihenfolge explizit festlegen willst:
    super_labels=[
        "macro_econ",
        "jobs_labor",
        "business_finance",
        "cost_of_living_housing",
        "healthcare_policy",
        "public_health_crises",
        "substances_gambling",
        "family_demographics",
        "law_crime_rights",
        "institutions_elections",
        "foreign_affairs_security",
        "environment_energy_infra",
        "media_meta",
    ]
)

# ========= 2) Pfade & Modell-ID =========

base_model_id = "deepseek-ai/deepseek-llm-7b-base"
train_path = "../2_Finetuning_Masterthesis/LIAR/preprocessed_train_cleaned_binary.csv"
test_path = "../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"
output_dir = "adapters/deepseek-liar-statement-domain-lora_12Classes"

# ========= 3) Training starten =========

if __name__ == "__main__":
    # Train
    train_domain_classifier(
        train_path=train_path,
        domain_config=domain_config,
        base_model_id=base_model_id,
        output_dir=output_dir,
        num_train_epochs=3,
        learning_rate=5e-4,
        per_device_train_batch_size=4,
        max_length=256,
    )

    # Eval
    evaluate_on_testset(
        test_path=test_path,
        domain_config=domain_config,
        base_model_id=base_model_id,
        lora_dir=output_dir,
    )


Super label distribution (train):
super_domain
macro_econ                  2606
institutions_elections      1798
law_crime_rights            1245
healthcare_policy           1087
jobs_labor                   764
foreign_affairs_security     651
media_meta                   571
environment_energy_infra     468
family_demographics          317
cost_of_living_housing       295
business_finance             189
public_health_crises         139
substances_gambling          110
Name: count, dtype: int64


Map:   0%|          | 0/10240 [00:00<?, ? examples/s]

Example tokenized sample keys: dict_keys(['input_ids', 'attention_mask'])
Input length: 196


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.


...

Step,Training Loss
50,0.523100
100,0.320400
150,0.310600
200,0.299300
250,0.298700
300,0.303800
350,0.296200
400,0.299800
450,0.301000
500,0.310100


✅ LoRA adapter & tokenizer saved to: adapters/deepseek-liar-statement-domain-lora_12Classes
Test set size: 1267
True label distribution (test):
super_domain_true
macro_econ                  316
institutions_elections      256
law_crime_rights            131
healthcare_policy           114
jobs_labor                  107
foreign_affairs_security     81
media_meta                   79
environment_energy_infra     61
cost_of_living_housing       44
family_demographics          28
business_finance             23
public_health_crises         14
substances_gambling          13
Name: count, dtype: int64


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🧠 Routing statements: 100%|██████████| 1267/1267 [09:15<00:00,  2.28it/s]

Predicted label distribution (test):
super_domain_pred
macro_econ                  345
institutions_elections      261
law_crime_rights            142
healthcare_policy           131
foreign_affairs_security     84
jobs_labor                   79
media_meta                   71
environment_energy_infra     66
family_demographics          32
substances_gambling          19
cost_of_living_housing       19
business_finance             12
public_health_crises          6
Name: count, dtype: int64

Accuracy (super_domain) on test set: 0.686

Classification report (per superlabel):
                          precision    recall  f1-score   support

              macro_econ       0.72      0.79      0.76       316
              jobs_labor       0.53      0.39      0.45       107
        business_finance       0.42      0.22      0.29        23
  cost_of_living_housing       0.58      0.25      0.35        44
       healthcare_policy       0.76      0.87      0.81       114
    public_health_cri

### Train-Experts

In [16]:
import pandas as pd
SUPER_LABELS = [
        "macro_econ",
        "jobs_labor",
        "business_finance",
        "cost_of_living_housing",
        "healthcare_policy",
        "public_health_crises",
        "substances_gambling",
        "family_demographics",
        "law_crime_rights",
        "institutions_elections",
        "foreign_affairs_security",
        "environment_energy_infra",
        "media_meta",
    ]
def map_subjects_to_super(subjects_str: str) -> str:
    """
    Map LIAR 'subjects' (comma-separated topics) to exactly one super label.
    """
    if not isinstance(subjects_str, str) or not subjects_str.strip():
        return "misc"

    domains = [s.strip() for s in subjects_str.split(",") if s.strip()]
    super_labels = {domain_to_super_12.get(d, "misc") for d in domains}

    # priorisierte Reihenfolge: erstes vorkommendes Label aus SUPER_LABELS
    for label in SUPER_LABELS:
        if label in super_labels:
            return label
    return "misc"

data_path = "../2_Finetuning_Masterthesis/LIAR/preprocessed_train_cleaned_binary.csv"  # anpassen falls nötig

df_all = pd.read_csv(
    data_path,
    sep="\t",
    quotechar='"',
    engine="python",
    dtype=str,
)

# super_domain aus subjects erzeugen
df_all["super_domain"] = df_all["subjects"].apply(map_subjects_to_super)

# Aufräumen
df_all = df_all.dropna(subset=["statement", "label", "super_domain"])
df_all = df_all[df_all["statement"].str.strip() != ""]
df_all = df_all[df_all["label"].isin(["True", "False"])]

print("Super-Domain-Verteilung:")
print(df_all["super_domain"].value_counts())


Super-Domain-Verteilung:
super_domain
macro_econ                  2361
institutions_elections      1798
law_crime_rights            1245
healthcare_policy           1087
jobs_labor                   764
foreign_affairs_security     651
media_meta                   571
environment_energy_infra     468
family_demographics          317
cost_of_living_housing       295
misc                         245
business_finance             189
public_health_crises         139
substances_gambling          110
Name: count, dtype: int64


In [17]:
df_all.to_csv("Results/preprocessed_train_cleaned_binary_with_super_domain_12.csv", sep="\t", index=False)


In [18]:
import pandas as pd

data_path = "Results/preprocessed_train_cleaned_binary_with_super_domain_12.csv"

df_all = pd.read_csv(
    data_path,
    sep="\t",
    quotechar='"',
    engine="python",
    dtype=str,
)

# Sicherstellen, dass wir nur sinnvolle Zeilen haben
df_all = df_all.dropna(subset=["statement", "label", "super_domain"])
df_all = df_all[df_all["statement"].str.strip() != ""]
df_all = df_all[df_all["label"].isin(["True", "False"])]

print("Gesamtverteilung super_domain:")
print(df_all["super_domain"].value_counts())

SUPER_LABELS = [
        "macro_econ",
        "jobs_labor",
        "business_finance",
        "cost_of_living_housing",
        "healthcare_policy",
        "public_health_crises",
        "substances_gambling",
        "family_demographics",
        "law_crime_rights",
        "institutions_elections",
        "foreign_affairs_security",
        "environment_energy_infra",
        "media_meta",
]

# Pro Domain ein DataFrame
domain_dfs = {
    domain: df_all[df_all["super_domain"] == domain].reset_index(drop=True)
    for domain in SUPER_LABELS
}

for dom, df_dom in domain_dfs.items():
    print(f"\nDomain: {dom} -> {len(df_dom)} Beispiele")


Gesamtverteilung super_domain:
super_domain
macro_econ                  2361
institutions_elections      1798
law_crime_rights            1245
healthcare_policy           1087
jobs_labor                   764
foreign_affairs_security     651
media_meta                   571
environment_energy_infra     468
family_demographics          317
cost_of_living_housing       295
misc                         245
business_finance             189
public_health_crises         139
substances_gambling          110
Name: count, dtype: int64

Domain: macro_econ -> 2361 Beispiele

Domain: jobs_labor -> 764 Beispiele

Domain: business_finance -> 189 Beispiele

Domain: cost_of_living_housing -> 295 Beispiele

Domain: healthcare_policy -> 1087 Beispiele

Domain: public_health_crises -> 139 Beispiele

Domain: substances_gambling -> 110 Beispiele

Domain: family_demographics -> 317 Beispiele

Domain: law_crime_rights -> 1245 Beispiele

Domain: institutions_elections -> 1798 Beispiele

Domain: foreign_affair

In [19]:
# run_train_experts.py (oder Notebook-Zelle)
import sys, os
import pandas as pd

# falls du in Jupyter bist:
sys.path.append(os.getcwd())

from train_expert_expert_config import ExpertConfig
from train_expert_expert_trainer import train_all_experts

# 1) Trainingsdaten laden – EIN gemeinsamer DataFrame
train_path = "Results/preprocessed_train_cleaned_binary_with_super_domain_12.csv"

df_train = pd.read_csv(
    train_path,
    sep="\t",
    quotechar='"',
    engine="python",
    dtype=str,
)

# 2) Konfiguration der Experten (Superdomains + Spaltennamen)
config = ExpertConfig(
    super_domains=[
        "macro_econ",
        "jobs_labor",
        "business_finance",
        "cost_of_living_housing",
        "healthcare_policy",
        "public_health_crises",
        "substances_gambling",
        "family_demographics",
        "law_crime_rights",
        "institutions_elections",
        "foreign_affairs_security",
        "environment_energy_infra",
        "media_meta",
    ],
    base_model_id="meta-llama/Llama-3.1-8B-Instruct",
    text_column="statement",
    label_column="label",
    domain_column="super_domain",
    max_length=256,
    num_train_epochs=3,
    learning_rate=5e-4,
    per_device_train_batch_size=4,
)

# 3) Output-Root – dort bekommt jede Domain seinen eigenen Unterordner
output_root = "adapters/Llama_experts_12Classes"

saved_paths = train_all_experts(
    df=df_train,
    config=config,
    output_root=output_root,
)

print("\nFertig! Modelle gespeichert unter:")
for dom, p in saved_paths.items():
    print(f"  {dom}: {p}")



=== Training expert for domain: macro_econ ===
Samples: 2361


Map:   0%|          | 0/2361 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 20,971,520 || all params: 8,051,232,768 || trainable%: 0.2605


🚀 Training:   0%|          | 0/888 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 20,971,520 || all params: 8,051,232,768 || trainable%: 0.2605


🚀 Training:   0%|          | 0/216 [00:00<?, ?it/s]

Step,Training Loss
25,0.811000
50,0.377400
75,0.343600
100,0.289900
125,0.259900
150,0.240600
175,0.146500
200,0.139800


✔️ Expert-LoRA for 'media_meta' saved to adapters/Llama_experts_12Classes/media_meta

Fertig! Modelle gespeichert unter:
  macro_econ: adapters/Llama_experts_12Classes/macro_econ
  jobs_labor: adapters/Llama_experts_12Classes/jobs_labor
  business_finance: adapters/Llama_experts_12Classes/business_finance
  cost_of_living_housing: adapters/Llama_experts_12Classes/cost_of_living_housing
  healthcare_policy: adapters/Llama_experts_12Classes/healthcare_policy
  public_health_crises: adapters/Llama_experts_12Classes/public_health_crises
  substances_gambling: adapters/Llama_experts_12Classes/substances_gambling
  family_demographics: adapters/Llama_experts_12Classes/family_demographics
  law_crime_rights: adapters/Llama_experts_12Classes/law_crime_rights
  institutions_elections: adapters/Llama_experts_12Classes/institutions_elections
  foreign_affairs_security: adapters/Llama_experts_12Classes/foreign_affairs_security
  environment_energy_infra: adapters/Llama_experts_12Classes/environmen

In [20]:
# run_train_experts.py (oder Notebook-Zelle)
import sys, os
import pandas as pd

# falls du in Jupyter bist:
sys.path.append(os.getcwd())

from train_expert_expert_config import ExpertConfig
from train_expert_expert_trainer import train_all_experts

# 1) Trainingsdaten laden – EIN gemeinsamer DataFrame
train_path = "Results/preprocessed_train_cleaned_binary_with_super_domain_12.csv"

df_train = pd.read_csv(
    train_path,
    sep="\t",
    quotechar='"',
    engine="python",
    dtype=str,
)

# 2) Konfiguration der Experten (Superdomains + Spaltennamen)
config = ExpertConfig(
    super_domains=[
        "macro_econ",
        "jobs_labor",
        "business_finance",
        "cost_of_living_housing",
        "healthcare_policy",
        "public_health_crises",
        "substances_gambling",
        "family_demographics",
        "law_crime_rights",
        "institutions_elections",
        "foreign_affairs_security",
        "environment_energy_infra",
        "media_meta",
    ],
    base_model_id="google/gemma-7b-it",
    text_column="statement",
    label_column="label",
    domain_column="super_domain",
    max_length=256,
    num_train_epochs=3,
    learning_rate=5e-4,
    per_device_train_batch_size=4,
)

# 3) Output-Root – dort bekommt jede Domain seinen eigenen Unterordner
output_root = "adapters/gemma_experts_12Classes"

saved_paths = train_all_experts(
    df=df_train,
    config=config,
    output_root=output_root,
)

print("\nFertig! Modelle gespeichert unter:")
for dom, p in saved_paths.items():
    print(f"  {dom}: {p}")



=== Training expert for domain: macro_econ ===
Samples: 2361


Map:   0%|          | 0/2361 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 25,001,984 || all params: 8,562,682,880 || trainable%: 0.2920


🚀 Training:   0%|          | 0/888 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

trainable params: 25,001,984 || all params: 8,562,682,880 || trainable%: 0.2920


🚀 Training:   0%|          | 0/216 [00:00<?, ?it/s]

Step,Training Loss
25,2.363300
50,0.699400
75,0.631800
100,0.489400
125,0.449900
150,0.382100
175,0.252800
200,0.236000


✔️ Expert-LoRA for 'media_meta' saved to adapters/gemma_experts_12Classes/media_meta

Fertig! Modelle gespeichert unter:
  macro_econ: adapters/gemma_experts_12Classes/macro_econ
  jobs_labor: adapters/gemma_experts_12Classes/jobs_labor
  business_finance: adapters/gemma_experts_12Classes/business_finance
  cost_of_living_housing: adapters/gemma_experts_12Classes/cost_of_living_housing
  healthcare_policy: adapters/gemma_experts_12Classes/healthcare_policy
  public_health_crises: adapters/gemma_experts_12Classes/public_health_crises
  substances_gambling: adapters/gemma_experts_12Classes/substances_gambling
  family_demographics: adapters/gemma_experts_12Classes/family_demographics
  law_crime_rights: adapters/gemma_experts_12Classes/law_crime_rights
  institutions_elections: adapters/gemma_experts_12Classes/institutions_elections
  foreign_affairs_security: adapters/gemma_experts_12Classes/foreign_affairs_security
  environment_energy_infra: adapters/gemma_experts_12Classes/environmen

In [21]:
# run_train_experts.py (oder Notebook-Zelle)
import sys, os
import pandas as pd

# falls du in Jupyter bist:
sys.path.append(os.getcwd())

from train_expert_expert_config import ExpertConfig
from train_expert_expert_trainer import train_all_experts

# 1) Trainingsdaten laden – EIN gemeinsamer DataFrame
train_path = "Results/preprocessed_train_cleaned_binary_with_super_domain_12.csv"

df_train = pd.read_csv(
    train_path,
    sep="\t",
    quotechar='"',
    engine="python",
    dtype=str,
)

# 2) Konfiguration der Experten (Superdomains + Spaltennamen)
config = ExpertConfig(
    super_domains=[
        "macro_econ",
        "jobs_labor",
        "business_finance",
        "cost_of_living_housing",
        "healthcare_policy",
        "public_health_crises",
        "substances_gambling",
        "family_demographics",
        "law_crime_rights",
        "institutions_elections",
        "foreign_affairs_security",
        "environment_energy_infra",
        "media_meta",
    ],
    base_model_id="deepseek-ai/deepseek-llm-7b-base",
    text_column="statement",
    label_column="label",
    domain_column="super_domain",
    max_length=256,
    num_train_epochs=3,
    learning_rate=5e-4,
    per_device_train_batch_size=4,
)

# 3) Output-Root – dort bekommt jede Domain seinen eigenen Unterordner
output_root = "adapters/deepseek_experts_12Classes"

saved_paths = train_all_experts(
    df=df_train,
    config=config,
    output_root=output_root,
)

print("\nFertig! Modelle gespeichert unter:")
for dom, p in saved_paths.items():
    print(f"  {dom}: {p}")



=== Training expert for domain: macro_econ ===
Samples: 2361


Map:   0%|          | 0/2361 [00:00<?, ? examples/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 18,739,200 || all params: 6,929,104,896 || trainable%: 0.2704


🚀 Training:   0%|          | 0/888 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 18,739,200 || all params: 6,929,104,896 || trainable%: 0.2704


🚀 Training:   0%|          | 0/216 [00:00<?, ?it/s]

Step,Training Loss
25,0.728100
50,0.406100
75,0.371100
100,0.364400
125,0.332000
150,0.329300
175,0.264500
200,0.250700


✔️ Expert-LoRA for 'media_meta' saved to adapters/deepseek_experts_12Classes/media_meta

Fertig! Modelle gespeichert unter:
  macro_econ: adapters/deepseek_experts_12Classes/macro_econ
  jobs_labor: adapters/deepseek_experts_12Classes/jobs_labor
  business_finance: adapters/deepseek_experts_12Classes/business_finance
  cost_of_living_housing: adapters/deepseek_experts_12Classes/cost_of_living_housing
  healthcare_policy: adapters/deepseek_experts_12Classes/healthcare_policy
  public_health_crises: adapters/deepseek_experts_12Classes/public_health_crises
  substances_gambling: adapters/deepseek_experts_12Classes/substances_gambling
  family_demographics: adapters/deepseek_experts_12Classes/family_demographics
  law_crime_rights: adapters/deepseek_experts_12Classes/law_crime_rights
  institutions_elections: adapters/deepseek_experts_12Classes/institutions_elections
  foreign_affairs_security: adapters/deepseek_experts_12Classes/foreign_affairs_security
  environment_energy_infra: adapter

### Evaluation

In [9]:
import pandas as pd
SUPER_LABELS = [
        "macro_econ",
        "jobs_labor",
        "business_finance",
        "cost_of_living_housing",
        "healthcare_policy",
        "public_health_crises",
        "substances_gambling",
        "family_demographics",
        "law_crime_rights",
        "institutions_elections",
        "foreign_affairs_security",
        "environment_energy_infra",
        "media_meta",
]
def map_subjects_to_super(subjects_str: str) -> str:
    """
    Map LIAR 'subjects' (comma-separated topics) to exactly one super label.
    """
    if not isinstance(subjects_str, str) or not subjects_str.strip():
        return "misc"

    domains = [s.strip() for s in subjects_str.split(",") if s.strip()]
    super_labels = {domain_to_super_12.get(d, "misc") for d in domains}

    # priorisierte Reihenfolge: erstes vorkommendes Label aus SUPER_LABELS
    for label in SUPER_LABELS:
        if label in super_labels:
            return label
    return "misc"

data_path = "../1_Basics_Masterthesis/LIAR/preprocessed_test_cleaned_binary.csv"  # anpassen falls nötig

df_all = pd.read_csv(
    data_path,
    sep="\t",
    quotechar='"',
    engine="python",
    dtype=str,
)

# super_domain aus subjects erzeugen
df_all["super_domain"] = df_all["subjects"].apply(map_subjects_to_super)

# Aufräumen
df_all = df_all.dropna(subset=["statement", "label", "super_domain"])
df_all = df_all[df_all["statement"].str.strip() != ""]
df_all = df_all[df_all["label"].isin(["True", "False"])]

print("Super-Domain-Verteilung:")
print(df_all["super_domain"].value_counts())
df_all.to_csv("Results/preprocessed_test_cleaned_binary_with_super_domain_12.csv", sep="\t", index=False)



Super-Domain-Verteilung:
super_domain
macro_econ                  279
institutions_elections      256
law_crime_rights            131
healthcare_policy           114
jobs_labor                  107
foreign_affairs_security     81
media_meta                   79
environment_energy_infra     61
cost_of_living_housing       44
misc                         37
family_demographics          28
business_finance             23
public_health_crises         14
substances_gambling          13
Name: count, dtype: int64


In [10]:
from multiagent_pipeline import run_full_pipeline

SUPER_LABELS_12 = [
        "macro_econ",
        "jobs_labor",
        "business_finance",
        "cost_of_living_housing",
        "healthcare_policy",
        "public_health_crises",
        "substances_gambling",
        "family_demographics",
        "law_crime_rights",
        "institutions_elections",
        "foreign_affairs_security",
        "environment_energy_infra",
        "media_meta",
]

df_pred = run_full_pipeline(
    base_model_id="meta-llama/Llama-3.1-8B-Instruct",
    router_lora_dir="adapters/llama-liar-statement-domain-lora_12Classes",
    expert_root="adapters/Llama_experts_12Classes",
    super_labels=SUPER_LABELS_12,
    test_path="Results/preprocessed_test_cleaned_binary_with_super_domain_12.csv",
    out_path="Results/Llama_test_predictions_with_experts_12.csv",
    text_col="statement",
    label_col="label",
    domain_col="super_domain",
    sep="\t",
)


Test set size: 1267
True domain distribution:
super_domain
macro_econ                  279
institutions_elections      256
law_crime_rights            131
healthcare_policy           114
jobs_labor                  107
foreign_affairs_security     81
media_meta                   79
environment_energy_infra     61
cost_of_living_housing       44
misc                         37
family_demographics          28
business_finance             23
public_health_crises         14
substances_gambling          13
Name: count, dtype: int64

=== Phase 1: Routing (Statement -> super_domain) ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🌍 Routing-only: 100%|██████████| 1267/1267 [10:34<00:00,  2.00it/s]



Predicted domain distribution (router):
domain_pred_router
macro_econ                  352
institutions_elections      249
healthcare_policy           133
law_crime_rights            129
foreign_affairs_security     90
media_meta                   83
jobs_labor                   67
environment_energy_infra     67
family_demographics          32
cost_of_living_housing       23
substances_gambling          16
business_finance             14
public_health_crises         12
Name: count, dtype: int64


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: environment_energy_infra: 100%|██████████| 67/67 [14:58<00:00, 13.41s/it]


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: media_meta: 100%|██████████| 83/83 [12:34<00:00,  9.08s/it]


Sample rows with predictions:
                                           statement            super_domain  \
0  Building a wall on the U.S . Mexico border wil...        law_crime_rights   
1  Wisconsin is on pace to double the number of l...              jobs_labor   
2  Says John McCain has done nothing to help the ...     family_demographics   
3  Suzanne Bonamici supports a plan that will cut...       healthcare_policy   
4  When asked by a reporter whether hes at the ce...  institutions_elections   

              domain_pred  label verdict_pred  
0        law_crime_rights   True         True  
1              jobs_labor  False         True  
2     family_demographics  False        False  
3       healthcare_policy   True        False  
4  institutions_elections  False         True  

🌍 Domain routing accuracy: 0.689
Confusion matrix (rows=true, cols=pred):
[[232  14   0   4   8   0   1   2   0  10   1   5   2]
 [ 31  41   1   0   4   2   1   0   1  23   0   0   3]
 [  8   0   5  

In [ ]:
from multiagent_pipeline import run_full_pipeline

SUPER_LABELS_12 = [
        "macro_econ",
        "jobs_labor",
        "business_finance",
        "cost_of_living_housing",
        "healthcare_policy",
        "public_health_crises",
        "substances_gambling",
        "family_demographics",
        "law_crime_rights",
        "institutions_elections",
        "foreign_affairs_security",
        "environment_energy_infra",
        "media_meta",
]

df_pred = run_full_pipeline(
    base_model_id="google/gemma-7b-it",
    router_lora_dir="adapters/gemma-liar-statement-domain-lora_12Classes",
    expert_root="adapters/gemma_experts_12Classes",
    super_labels=SUPER_LABELS_12,
    test_path="Results/preprocessed_test_cleaned_binary_with_super_domain_12.csv",
    out_path="Results/gemma_test_predictions_with_experts_12.csv",
    text_col="statement",
    label_col="label",
    domain_col="super_domain",
    sep="\t",
)


Test set size: 1267
True domain distribution:
super_domain
macro_econ                  279
institutions_elections      256
law_crime_rights            131
healthcare_policy           114
jobs_labor                  107
foreign_affairs_security     81
media_meta                   79
environment_energy_infra     61
cost_of_living_housing       44
misc                         37
family_demographics          28
business_finance             23
public_health_crises         14
substances_gambling          13
Name: count, dtype: int64

=== Phase 1: Routing (Statement -> super_domain) ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🌍 Routing-only: 100%|██████████| 1267/1267 [05:32<00:00,  3.81it/s]



Predicted domain distribution (router):
domain_pred_router
macro_econ                  274
institutions_elections      200
law_crime_rights            144
healthcare_policy           123
jobs_labor                  116
media_meta                  104
foreign_affairs_security     84
environment_energy_infra     73
cost_of_living_housing       48
family_demographics          40
business_finance             34
public_health_crises         17
substances_gambling          10
Name: count, dtype: int64


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: macro_econ: 100%|██████████| 274/274 [1:47:30<00:00, 23.54s/it]  


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: jobs_labor:   2%|▏         | 2/116 [00:59<51:49, 27.28s/it]  

In [ ]:
from multiagent_pipeline import run_full_pipeline

SUPER_LABELS_12 = [
        "macro_econ",
        "jobs_labor",
        "business_finance",
        "cost_of_living_housing",
        "healthcare_policy",
        "public_health_crises",
        "substances_gambling",
        "family_demographics",
        "law_crime_rights",
        "institutions_elections",
        "foreign_affairs_security",
        "environment_energy_infra",
        "media_meta",
]

df_pred = run_full_pipeline(
    base_model_id="deepseek-ai/deepseek-llm-7b-base",
    router_lora_dir="adapters/deepseek-liar-statement-domain-lora_12Classes",
    expert_root="adapters/deepseek_experts_12Classes",
    super_labels=SUPER_LABELS_12,
    test_path="Results/preprocessed_test_cleaned_binary_with_super_domain_12.csv",
    out_path="Results/deepseek_test_predictions_with_experts_12.csv",
    text_col="statement",
    label_col="label",
    domain_col="super_domain",
    sep="\t",
)


Test set size: 1267
True domain distribution:
super_domain
macro_econ                  279
institutions_elections      256
law_crime_rights            131
healthcare_policy           114
jobs_labor                  107
foreign_affairs_security     81
media_meta                   79
environment_energy_infra     61
cost_of_living_housing       44
misc                         37
family_demographics          28
business_finance             23
public_health_crises         14
substances_gambling          13
Name: count, dtype: int64

=== Phase 1: Routing (Statement -> super_domain) ===


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🌍 Routing-only: 100%|██████████| 1267/1267 [09:18<00:00,  2.27it/s]



Predicted domain distribution (router):
domain_pred_router
macro_econ                  352
institutions_elections      242
law_crime_rights            145
healthcare_policy           136
foreign_affairs_security     85
media_meta                   80
jobs_labor                   72
environment_energy_infra     71
family_demographics          29
substances_gambling          18
cost_of_living_housing       18
business_finance             13
public_health_crises          6
Name: count, dtype: int64


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: macro_econ:   1%|▏         | 5/352 [00:35<39:37,  6.85s/it]

## Evalutation on Ground Truth Labels

In [1]:
import importlib
import evaluate_experts_all
importlib.reload(evaluate_experts_all)
evaluate_experts_all.main()

2026-02-28 09:39:34.117794: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.




######################################################################
  Experts: backbone=llama  k=5
  Root: ./adapters/Llama_experts_5Classes
######################################################################
  Full test set size: 1267

  Domain: socioeconomic_policy  |  n=616
  Label distribution: {'True': 348, 'False': 268}


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

llama k=5 socioeconomic_policy: 100%|██████████| 616/616 [01:32<00:00,  6.64it/s]


...

deepseek k=12 environment_energy_infra: 100%|██████████| 61/61 [00:13<00:00,  4.43it/s]



  Accuracy: 0.4918
              precision    recall  f1-score   support

        True       0.49      1.00      0.66        30
       False       0.00      0.00      0.00        31

    accuracy                           0.49        61
   macro avg       0.25      0.50      0.33        61
weighted avg       0.24      0.49      0.32        61

  Confusion matrix [True, False]:
[[30  0]
 [31  0]]

  Domain: media_meta  |  n=79
  Label distribution: {'True': 46, 'False': 33}


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

deepseek k=12 media_meta: 100%|██████████| 79/79 [00:17<00:00,  4.45it/s]


  Accuracy: 0.5823
              precision    recall  f1-score   support

        True       0.59      0.93      0.72        46
       False       0.50      0.09      0.15        33

    accuracy                           0.58        79
   macro avg       0.54      0.51      0.44        79
weighted avg       0.55      0.58      0.49        79

  Confusion matrix [True, False]:
[[43  3]
 [30  3]]

===== Gesamt-Performance über alle Domains =====
Backbone=DEEPSEEK  k=12  |  total_samples=1230

Overall accuracy: 0.613

Overall classification report:
              precision    recall  f1-score   support

        True       0.60      0.93      0.73       689
       False       0.70      0.21      0.32       541

    accuracy                           0.61      1230
   macro avg       0.65      0.57      0.52      1230
weighted avg       0.65      0.61      0.55      1230

Overall confusion matrix (True, False):
[[642  47]
 [429 112]]

Per-domain accuracy summary:
  macro_econ              

## Conclusion

In [1]:
from evaluate_domainrouter import evaluate_all_verbose

SUPER_LABELS_5 = [        
        "socioeconomic_policy",
        "foreign_security",
        "governance_law",
        "environment_science",
        "society_culture",
        "misc",
        ]
SUPER_LABELS_8 = [        
        "economy",
        "health_social",
        "foreign_security",
        "law_rights",
        "politics_government",
        "environment_energy",
        "society_culture",
        "misc",
        ]
SUPER_LABELS_12 = [
        "macro_econ",
        "jobs_labor",
        "business_finance",
        "cost_of_living_housing",
        "healthcare_policy",
        "public_health_crises",
        "substances_gambling",
        "family_demographics",
        "law_crime_rights",
        "institutions_elections",
        "foreign_affairs_security",
        "environment_energy_infra",
        "media_meta",
    ]

summary = evaluate_all_verbose(
    folder="Results",
    super_labels_by_k={5: SUPER_LABELS_5, 8: SUPER_LABELS_8, 12: SUPER_LABELS_12},
)
summary



Llama_test_predictions_with_experts_12.csv   |   model=llama   |   k=12
Test set size: 1267
True domain distribution:
super_domain
macro_econ                  279
institutions_elections      256
law_crime_rights            131
healthcare_policy           114
jobs_labor                  107
foreign_affairs_security     81
media_meta                   79
environment_energy_infra     61
cost_of_living_housing       44
misc                         37
family_demographics          28
business_finance             23
public_health_crises         14
substances_gambling          13
Name: count, dtype: int64

Predicted domain distribution (router):
domain_pred_router
macro_econ                  352
institutions_elections      249
healthcare_policy           133
law_crime_rights            129
foreign_affairs_security     90
media_meta                   83
jobs_labor                   67
environment_energy_infra     67
family_demographics          32
cost_of_living_housing       23
substances_gam

,file,model,k,mode,n_eval,accuracy,precision_true,recall_true,f1_true,confusion_matrix,domain_acc,n_domain_eval,domain_confusion_matrix
4,deepseek_test_predictions_with_experts_5.csv,deepseek,5,deepseek_k5,1267,0.660616,0.649160,0.865546,0.741897,[[618 96]\n [334 219]],0.787687,1267,[[538 2 63 7 6 0]\n [ 8 99 12 0 ...
5,deepseek_test_predictions_with_experts_8.csv,deepseek,8,deepseek_k8,1267,0.610103,0.596831,0.949580,0.732973,[[678 36]\n [458 95]],0.744278,1267,[[388 27 1 9 30 6 8 4]\n [ 8 118 ...
3,deepseek_test_predictions_with_experts_12.csv,deepseek,12,deepseek_k12,1267,0.609313,0.599818,0.921569,0.726670,[[658 56]\n [439 114]],0.680488,1230,[[228 16 2 3 7 0 1 2 1 9 1 ...
7,gemma_test_predictions_with_experts_5.csv,gemma,5,gemma_k5,1267,0.488556,0.585052,0.317927,0.411978,[[227 487]\n [161 392]],0.768745,1267,[[518 3 66 12 16 1]\n [ 8 94 13 5 ...
8,gemma_test_predictions_with_experts_8.csv,gemma,8,gemma_k8,1267,0.576953,0.610422,0.689076,0.647368,[[492 222]\n [314 239]],0.719811,1267,[[388 31 2 13 17 13 8 1]\n [ 8 115 ...
6,gemma_test_predictions_with_experts_12.csv,gemma,12,gemma_k12,1267,0.592739,0.618705,0.722689,0.666667,[[516 198]\n [318 235]],0.642276,1230,[[194 36 5 5 8 0 1 4 2 7 1 ...
1,Llama_test_predictions_with_experts_5.csv,llama,5,llama_k5,1267,0.543015,0.728814,0.301120,0.426165,[[215 499]\n [ 80 473]],0.765588,1267,[[547 3 48 5 11 2]\n [ 16 91 11 2 ...
2,Llama_test_predictions_with_experts_8.csv,llama,8,llama_k8,1267,0.632202,0.668478,0.689076,0.678621,[[492 222]\n [244 309]],0.739542,1267,[[393 28 2 7 29 3 8 3]\n [ 15 108 ...
0,Llama_test_predictions_with_experts_12.csv,llama,12,llama_k12,1267,0.635359,0.676471,0.676471,0.676471,[[483 231]\n [231 322]],0.688618,1230,[[232 14 0 4 8 0 1 2 0 10 1 ...


# Domainrouting + Checkability

### 5 Superlabels

In [4]:
from multiagent_pipeline_checkability import run_full_pipeline

SUPER_LABELS_5 = [
        "socioeconomic_policy",
        "foreign_security",
        "governance_law",
        "environment_science",
        "society_culture",
        "misc",
]

df_pred = run_full_pipeline(
    base_model_id="meta-llama/Llama-3.1-8B-Instruct",
    router_lora_dir="adapters/llama-liar-statement-domain-lora_5Classes",
    expert_root="adapters/Llama_experts_5Classes",
    super_labels=SUPER_LABELS_5,
    test_path="Results/preprocessed_test_cleaned_binary_with_super_domain_5.csv",
    out_path="Results/Llama_test_predictions_with_experts_5_checkability.csv",
    text_col="statement",
    label_col="label",
    domain_col="super_domain",
    sep="\t",
    use_checkability_gate=True,
)


Test set size: 1267
True domain distribution:
super_domain
socioeconomic_policy    616
governance_law          378
foreign_security        122
society_culture          61
environment_science      57
misc                     33
Name: count, dtype: int64

=== Phase 0: Checkability Gate ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Checkability: 100%|██████████| 1267/1267 [07:40<00:00,  2.75it/s]



Checkability categories:
checkability_category
fact_checkable              1100
opinion_or_ambiguous         122
non_claim                     42
sensitive_selfharm             2
needs_additional_context       1
Name: count, dtype: int64

✅ fact_checkable: 1100 | excluded: 167

=== Phase 1: Routing (Statement -> super_domain) ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: society_culture: 100%|██████████| 53/53 [10:26<00:00, 11.82s/it]


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: misc: 100%|██████████| 11/11 [01:43<00:00,  9.37s/it]



Sample rows with predictions (evaluated subset):
                                           statement          super_domain  \
0  Building a wall on the U.S . Mexico border wil...        governance_law   
1  Wisconsin is on pace to double the number of l...  socioeconomic_policy   
3  Suzanne Bonamici supports a plan that will cut...  socioeconomic_policy   
4  When asked by a reporter whether hes at the ce...        governance_law   
5  Over the past five years the federal governmen...  socioeconomic_policy   

            domain_pred  label verdict_pred checkability_category  
0  socioeconomic_policy   True        False        fact_checkable  
1  socioeconomic_policy  False        False        fact_checkable  
3  socioeconomic_policy   True        False        fact_checkable  
4        governance_law  False         True        fact_checkable  
5  socioeconomic_policy   True        False        fact_checkable  

🌍 Domain routing accuracy: 0.770
Confusion matrix (rows=true, cols=pred)

In [5]:
from multiagent_pipeline_checkability import run_full_pipeline

SUPER_LABELS_5 = [
        "socioeconomic_policy",
        "foreign_security",
        "governance_law",
        "environment_science",
        "society_culture",
        "misc",
]

df_pred = run_full_pipeline(
    base_model_id="google/gemma-7b-it",
    router_lora_dir="adapters/gemma-liar-statement-domain-lora_5Classes",
    expert_root="adapters/gemma_experts_5Classes",
    super_labels=SUPER_LABELS_5,
    test_path="Results/preprocessed_test_cleaned_binary_with_super_domain_5.csv",
    out_path="Results/gemma_test_predictions_with_experts_5_checkability.csv",
    text_col="statement",
    label_col="label",
    domain_col="super_domain",
    use_checkability_gate=True,
)


Test set size: 1267
True domain distribution:
super_domain
socioeconomic_policy    616
governance_law          378
foreign_security        122
society_culture          61
environment_science      57
misc                     33
Name: count, dtype: int64

=== Phase 0: Checkability Gate ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Checkability: 100%|██████████| 1267/1267 [33:17<00:00,  1.58s/it]



Checkability categories:
checkability_category
fact_checkable              535
needs_additional_context    383
opinion_or_ambiguous        279
non_claim                    68
sensitive_selfharm            2
Name: count, dtype: int64

✅ fact_checkable: 535 | excluded: 732

=== Phase 1: Routing (Statement -> super_domain) ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: society_culture: 100%|██████████| 39/39 [14:34<00:00, 22.43s/it]


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: misc: 100%|██████████| 5/5 [01:37<00:00, 19.53s/it]



Sample rows with predictions (evaluated subset):
                                           statement          super_domain  \
1  Wisconsin is on pace to double the number of l...  socioeconomic_policy   
4  When asked by a reporter whether hes at the ce...        governance_law   
5  Over the past five years the federal governmen...  socioeconomic_policy   
6  Says that Tennessee law requires that schools ...  socioeconomic_policy   
9  We know that more than half of Hillary Clinton...      foreign_security   

            domain_pred  label verdict_pred checkability_category  
1  socioeconomic_policy  False        False        fact_checkable  
4        governance_law  False         True        fact_checkable  
5  socioeconomic_policy   True        False        fact_checkable  
6  socioeconomic_policy   True        False        fact_checkable  
9        governance_law  False         True        fact_checkable  

🌍 Domain routing accuracy: 0.776
Confusion matrix (rows=true, cols=pred)

In [11]:
from multiagent_pipeline_checkability_deepseek import run_full_pipeline

SUPER_LABELS_5 = [
        "socioeconomic_policy",
        "foreign_security",
        "governance_law",
        "environment_science",
        "society_culture",
        "misc",
]

df_pred = run_full_pipeline(
    base_model_id="deepseek-ai/deepseek-llm-7b-base",
    router_lora_dir="adapters/deepseek-liar-statement-domain-lora_5Classes",
    expert_root="adapters/deepseek_experts_5Classes",
    super_labels=SUPER_LABELS_5,
    test_path="Results/preprocessed_test_cleaned_binary_with_super_domain_5.csv",
    out_path="Results/deepseek_test_predictions_with_experts_5_checkability.csv",
    text_col="statement",
    label_col="label",
    domain_col="super_domain",
    sep="\t",
    use_checkability_gate=True,
)


Test set size: 1267
True domain distribution:
super_domain
socioeconomic_policy    616
governance_law          378
foreign_security        122
society_culture          61
environment_science      57
misc                     33
Name: count, dtype: int64

=== Phase 0: Checkability Gate ===


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🔎 Checkability: 100%|██████████| 1267/1267 [1:03:51<00:00,  3.02s/it]



Checkability categories:
checkability_category
fact_checkable          1263
opinion_or_ambiguous       2
non_claim                  2
Name: count, dtype: int64

✅ fact_checkable: 1263 | excluded: 4

=== Phase 1: Routing (Statement -> super_domain) ===


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: society_culture: 100%|██████████| 49/49 [09:14<00:00, 11.32s/it]


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: misc: 100%|██████████| 9/9 [00:57<00:00,  6.38s/it]



Sample rows with predictions (evaluated subset):
                                           statement          super_domain  \
0  Building a wall on the U.S . Mexico border wil...        governance_law   
1  Wisconsin is on pace to double the number of l...  socioeconomic_policy   
2  Says John McCain has done nothing to help the ...  socioeconomic_policy   
3  Suzanne Bonamici supports a plan that will cut...  socioeconomic_policy   
4  When asked by a reporter whether hes at the ce...        governance_law   

            domain_pred  label verdict_pred checkability_category  
0        governance_law   True         True        fact_checkable  
1  socioeconomic_policy  False         True        fact_checkable  
2  socioeconomic_policy  False        False        fact_checkable  
3  socioeconomic_policy   True        False        fact_checkable  
4        governance_law  False        False        fact_checkable  

🌍 Domain routing accuracy: 0.787
Confusion matrix (rows=true, cols=pred)

### 8 Superlabels

In [1]:
from multiagent_pipeline_checkability import run_full_pipeline

SUPER_LABELS_8 = [
        "economy",
        "health_social",
        "foreign_security",
        "law_rights",
        "politics_government",
        "environment_energy",
        "society_culture",
        "misc",
]

df_pred = run_full_pipeline(
    base_model_id="meta-llama/Llama-3.1-8B-Instruct",
    router_lora_dir="adapters/llama-liar-statement-domain-lora_8Classes",
    expert_root="adapters/Llama_experts_8Classes",
    super_labels=SUPER_LABELS_8,
    test_path="Results/preprocessed_test_cleaned_binary_with_super_domain_8.csv",
    out_path="Results/Llama_test_predictions_with_experts_8_checkability.csv",
    text_col="statement",
    label_col="label",
    domain_col="super_domain",
    sep="\t",
    use_checkability_gate=True,
)


Test set size: 1267
True domain distribution:
super_domain
economy                473
politics_government    233
law_rights             145
health_social          143
foreign_security       122
society_culture         61
environment_energy      56
misc                    34
Name: count, dtype: int64

=== Phase 0: Checkability Gate ===


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Checkability: 100%|██████████| 1267/1267 [07:12<00:00,  2.93it/s]



Checkability categories:
checkability_category
fact_checkable              1100
opinion_or_ambiguous         122
non_claim                     42
sensitive_selfharm             2
needs_additional_context       1
Name: count, dtype: int64

✅ fact_checkable: 1100 | excluded: 167

=== Phase 1: Routing (Statement -> super_domain) ===


...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: society_culture: 100%|██████████| 52/52 [22:20<00:00, 25.78s/it]


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: misc: 100%|██████████| 15/15 [02:32<00:00, 10.18s/it]



Sample rows with predictions (evaluated subset):
                                           statement   super_domain  \
0  Building a wall on the U.S . Mexico border wil...     law_rights   
1  Wisconsin is on pace to double the number of l...        economy   
3  Suzanne Bonamici supports a plan that will cut...  health_social   
4  When asked by a reporter whether hes at the ce...     law_rights   
5  Over the past five years the federal governmen...        economy   

     domain_pred  label verdict_pred checkability_category  
0     law_rights   True         True        fact_checkable  
1        economy  False         True        fact_checkable  
3  health_social   True         True        fact_checkable  
4        economy  False         True        fact_checkable  
5        economy   True         True        fact_checkable  

🌍 Domain routing accuracy: 0.744
Confusion matrix (rows=true, cols=pred):
[[350  24   2   6  29   2   4   2]
 [ 15  97   3  12   1   0   3   0]
 [  6   2  7

In [2]:
from multiagent_pipeline_checkability import run_full_pipeline

SUPER_LABELS_8 = [
        "economy",
        "health_social",
        "foreign_security",
        "law_rights",
        "politics_government",
        "environment_energy",
        "society_culture",
        "misc",
]

df_pred = run_full_pipeline(
    base_model_id="google/gemma-7b-it",
    router_lora_dir="adapters/gemma-liar-statement-domain-lora_8Classes",
    expert_root="adapters/gemma_experts_8Classes",
    super_labels=SUPER_LABELS_8,
    test_path="Results/preprocessed_test_cleaned_binary_with_super_domain_8.csv",
    out_path="Results/gemma_test_predictions_with_experts_8_checkability.csv",
    text_col="statement",
    label_col="label",
    domain_col="super_domain",
    use_checkability_gate=True,
)


Test set size: 1267
True domain distribution:
super_domain
economy                473
politics_government    233
law_rights             145
health_social          143
foreign_security       122
society_culture         61
environment_energy      56
misc                    34
Name: count, dtype: int64

=== Phase 0: Checkability Gate ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Checkability: 100%|██████████| 1267/1267 [34:13<00:00,  1.62s/it]



Checkability categories:
checkability_category
fact_checkable              535
needs_additional_context    383
opinion_or_ambiguous        279
non_claim                    68
sensitive_selfharm            2
Name: count, dtype: int64

✅ fact_checkable: 535 | excluded: 732

=== Phase 1: Routing (Statement -> super_domain) ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: society_culture: 100%|██████████| 35/35 [19:07<00:00, 32.79s/it]


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: misc: 100%|██████████| 5/5 [02:41<00:00, 32.30s/it]



Sample rows with predictions (evaluated subset):
                                           statement      super_domain  \
1  Wisconsin is on pace to double the number of l...           economy   
4  When asked by a reporter whether hes at the ce...        law_rights   
5  Over the past five years the federal governmen...           economy   
6  Says that Tennessee law requires that schools ...           economy   
9  We know that more than half of Hillary Clinton...  foreign_security   

        domain_pred  label verdict_pred checkability_category  
1           economy  False        False        fact_checkable  
4           economy  False        False        fact_checkable  
5           economy   True         True        fact_checkable  
6           economy   True         True        fact_checkable  
9  foreign_security  False         True        fact_checkable  

🌍 Domain routing accuracy: 0.738
Confusion matrix (rows=true, cols=pred):
[[188  11   2   9   7   4   2   1]
 [  4  53  

In [3]:
from multiagent_pipeline_checkability import run_full_pipeline

SUPER_LABELS_8 = [
        "economy",
        "health_social",
        "foreign_security",
        "law_rights",
        "politics_government",
        "environment_energy",
        "society_culture",
        "misc",
]

df_pred = run_full_pipeline(
    base_model_id="deepseek-ai/deepseek-llm-7b-base",
    router_lora_dir="adapters/deepseek-liar-statement-domain-lora_8Classes",
    expert_root="adapters/deepseek_experts_8Classes",
    super_labels=SUPER_LABELS_8,
    test_path="Results/preprocessed_test_cleaned_binary_with_super_domain_8.csv",
    out_path="Results/deepseek_test_predictions_with_experts_8_checkability.csv",
    text_col="statement",
    label_col="label",
    domain_col="super_domain",
    sep="\t",
    use_checkability_gate=True,
)


Test set size: 1267
True domain distribution:
super_domain
economy                473
politics_government    233
law_rights             145
health_social          143
foreign_security       122
society_culture         61
environment_energy      56
misc                    34
Name: count, dtype: int64

=== Phase 0: Checkability Gate ===


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🔎 Checkability: 100%|██████████| 1267/1267 [1:06:28<00:00,  3.15s/it]



Checkability categories:
checkability_category
opinion_or_ambiguous    1267
Name: count, dtype: int64

✅ fact_checkable: 0 | excluded: 1267


RuntimeError: No fact_checkable samples found. Gate too strict or data issue.

In [2]:
evaluate_predictions("Results/deepseek_test_predictions_with_experts_8.csv")



✅ Verdict Evaluation (overall)
Samples: 1267
Verdict Accuracy: 0.610

Classification report (True/False):
              precision    recall  f1-score   support

        True       0.60      0.95      0.73       714
       False       0.73      0.17      0.28       553

    accuracy                           0.61      1267
   macro avg       0.66      0.56      0.51      1267
weighted avg       0.65      0.61      0.53      1267

Confusion matrix [rows=true, cols=pred] (True, False):
[[678  36]
 [458  95]]

🌍 Domain Routing Evaluation
Routing samples: 1267
Domain-Routing Accuracy: 0.744
Domain-Routing Confusion Matrix (rows=true, cols=pred):
[[388  27   1   9  30   6   8   4]
 [  8 118   3  10   2   0   2   0]
 [  6   3  94  10   5   0   4   0]
 [  2   9  10 107  14   0   3   0]
 [ 43   3   8   6 155  10   7   1]
 [ 11   0   1   0   4  36   1   3]
 [  9   1   2   4  11   0  34   0]
 [ 15   2   1   0   4   1   0  11]]
Domain label order: ['economy', 'health_social', 'foreign_security', 

### 12 Superlabels

In [ ]:
from multiagent_pipeline_checkability import run_full_pipeline

SUPER_LABELS_12 = [
        "macro_econ",
        "jobs_labor",
        "business_finance",
        "cost_of_living_housing",
        "healthcare_policy",
        "public_health_crises",
        "substances_gambling",
        "family_demographics",
        "law_crime_rights",
        "institutions_elections",
        "foreign_affairs_security",
        "environment_energy_infra",
        "media_meta",
    ]

df_pred = run_full_pipeline(
    base_model_id="meta-llama/Llama-3.1-8B-Instruct",
    router_lora_dir="adapters/llama-liar-statement-domain-lora_12Classes",
    expert_root="adapters/Llama_experts_12Classes",
    super_labels=SUPER_LABELS_12,
    test_path="Results/preprocessed_test_cleaned_binary_with_super_domain_12.csv",
    out_path="Results/Llama_test_predictions_with_experts_12_checkability.csv",
    text_col="statement",
    label_col="label",
    domain_col="super_domain",
    sep="\t",
    use_checkability_gate=True,
)


Test set size: 1267
True domain distribution:
super_domain
macro_econ                  279
institutions_elections      256
law_crime_rights            131
healthcare_policy           114
jobs_labor                  107
foreign_affairs_security     81
media_meta                   79
environment_energy_infra     61
cost_of_living_housing       44
misc                         37
family_demographics          28
business_finance             23
public_health_crises         14
substances_gambling          13
Name: count, dtype: int64

=== Phase 0: Checkability Gate ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [10]:
from multiagent_pipeline_checkability import run_full_pipeline

SUPER_LABELS_12 = [
        "macro_econ",
        "jobs_labor",
        "business_finance",
        "cost_of_living_housing",
        "healthcare_policy",
        "public_health_crises",
        "substances_gambling",
        "family_demographics",
        "law_crime_rights",
        "institutions_elections",
        "foreign_affairs_security",
        "environment_energy_infra",
        "media_meta",
    ]

df_pred = run_full_pipeline(
    base_model_id="google/gemma-7b-it",
    router_lora_dir="adapters/gemma-liar-statement-domain-lora_12Classes",
    expert_root="adapters/gemma_experts_12Classes",
    super_labels=SUPER_LABELS_12,
    test_path="Results/preprocessed_test_cleaned_binary_with_super_domain_12.csv",
    out_path="Results/gemma_test_predictions_with_experts_12_checkability.csv",
    text_col="statement",
    label_col="label",
    domain_col="super_domain",
    use_checkability_gate=True,
)


Test set size: 1267
True domain distribution:
super_domain
macro_econ                  279
institutions_elections      256
law_crime_rights            131
healthcare_policy           114
jobs_labor                  107
foreign_affairs_security     81
media_meta                   79
environment_energy_infra     61
cost_of_living_housing       44
misc                         37
family_demographics          28
business_finance             23
public_health_crises         14
substances_gambling          13
Name: count, dtype: int64

=== Phase 0: Checkability Gate ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🔎 Checkability: 100%|██████████| 1267/1267 [33:31<00:00,  1.59s/it]



Checkability categories:
checkability_category
fact_checkable              535
needs_additional_context    383
opinion_or_ambiguous        279
non_claim                    68
sensitive_selfharm            2
Name: count, dtype: int64

✅ fact_checkable: 535 | excluded: 732

=== Phase 1: Routing (Statement -> super_domain) ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: environment_energy_infra: 100%|██████████| 30/30 [10:57<00:00, 21.92s/it]


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: media_meta: 100%|██████████| 39/39 [23:43<00:00, 36.50s/it]



Sample rows with predictions (evaluated subset):
                                           statement  \
1  Wisconsin is on pace to double the number of l...   
4  When asked by a reporter whether hes at the ce...   
5  Over the past five years the federal governmen...   
6  Says that Tennessee law requires that schools ...   
9  We know that more than half of Hillary Clinton...   

               super_domain             domain_pred  label verdict_pred  \
1                jobs_labor              jobs_labor  False        False   
4    institutions_elections  institutions_elections  False         True   
5    cost_of_living_housing  cost_of_living_housing   True         True   
6                macro_econ              macro_econ   True         True   
9  foreign_affairs_security  institutions_elections  False         True   

  checkability_category  
1        fact_checkable  
4        fact_checkable  
5        fact_checkable  
6        fact_checkable  
9        fact_checkable  

🌍 Dom

In [ ]:
from multiagent_pipeline_checkability_deepseek import run_full_pipeline

SUPER_LABELS_12 = [
        "macro_econ",
        "jobs_labor",
        "business_finance",
        "cost_of_living_housing",
        "healthcare_policy",
        "public_health_crises",
        "substances_gambling",
        "family_demographics",
        "law_crime_rights",
        "institutions_elections",
        "foreign_affairs_security",
        "environment_energy_infra",
        "media_meta",
    ]

df_pred = run_full_pipeline(
    base_model_id="deepseek-ai/deepseek-llm-7b-base",
    router_lora_dir="adapters/deepseek-liar-statement-domain-lora_12Classes",
    expert_root="adapters/deepseek_experts_12Classes",
    super_labels=SUPER_LABELS_12,
    test_path="Results/preprocessed_test_cleaned_binary_with_super_domain_12.csv",
    out_path="Results/deepseek_test_predictions_with_experts_12_checkability.csv",
    text_col="statement",
    label_col="label",
    domain_col="super_domain",
    sep="\t",
    use_checkability_gate=True,
)


Test set size: 1267
True domain distribution:
super_domain
macro_econ                  279
institutions_elections      256
law_crime_rights            131
healthcare_policy           114
jobs_labor                  107
foreign_affairs_security     81
media_meta                   79
environment_energy_infra     61
cost_of_living_housing       44
misc                         37
family_demographics          28
business_finance             23
public_health_crises         14
substances_gambling          13
Name: count, dtype: int64

=== Phase 0: Checkability Gate ===


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🔎 Checkability: 100%|██████████| 1267/1267 [1:03:42<00:00,  3.02s/it]



Checkability categories:
checkability_category
fact_checkable          1263
opinion_or_ambiguous       2
non_claim                  2
Name: count, dtype: int64

✅ fact_checkable: 1263 | excluded: 4

=== Phase 1: Routing (Statement -> super_domain) ===


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

🌍 Routing-only: 100%|██████████| 1263/1263 [09:43<00:00,  2.16it/s]



Predicted domain distribution (router):
domain_pred_router
macro_econ                  351
institutions_elections      242
law_crime_rights            145
healthcare_policy           134
foreign_affairs_security     85
media_meta                   80
jobs_labor                   72
environment_energy_infra     70
family_demographics          29
substances_gambling          18
cost_of_living_housing       18
business_finance             13
public_health_crises          6
Name: count, dtype: int64


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 Expert: macro_econ:   4%|▎         | 13/351 [01:29<37:58,  6.74s/it]

## Conclusion

In [1]:
from evaluate_domainrouter_checkability import evaluate_all_verbose

SUPER_LABELS_5 = [        
        "socioeconomic_policy",
        "foreign_security",
        "governance_law",
        "environment_science",
        "society_culture",
        "misc",
        ]
SUPER_LABELS_8 = [        
        "economy",
        "health_social",
        "foreign_security",
        "law_rights",
        "politics_government",
        "environment_energy",
        "society_culture",
        "misc",
        ]
SUPER_LABELS_12 = [
        "macro_econ",
        "jobs_labor",
        "business_finance",
        "cost_of_living_housing",
        "healthcare_policy",
        "public_health_crises",
        "substances_gambling",
        "family_demographics",
        "law_crime_rights",
        "institutions_elections",
        "foreign_affairs_security",
        "environment_energy_infra",
        "media_meta",
    ]

summary = evaluate_all_verbose(
    folder="Results",
    pattern="*test_predictions_with_experts*_checkability.csv",
    super_labels_by_k={5: SUPER_LABELS_5, 8: SUPER_LABELS_8, 12: SUPER_LABELS_12},
)
summary



Llama_test_predictions_with_experts_12_checkability.csv   |   model=llama   |   k=12
Filtered by checkability (fact_checkable): 1267 -> 1100
Test set size: 1100
True domain distribution:
super_domain
macro_econ                  247
institutions_elections      225
law_crime_rights            112
healthcare_policy           103
jobs_labor                   96
media_meta                   68
foreign_affairs_security     62
environment_energy_infra     46
cost_of_living_housing       40
misc                         31
family_demographics          26
business_finance             20
public_health_crises         12
substances_gambling          12
Name: count, dtype: int64

Predicted domain distribution (router):
domain_pred_router
macro_econ                  311
institutions_elections      222
healthcare_policy           118
law_crime_rights            111
foreign_affairs_security     71
media_meta                   67
jobs_labor                   62
environment_energy_infra     52
family_de

,file,model,k,mode,n_eval,accuracy,precision_true,recall_true,confusion_matrix,domain_acc,n_domain_eval,domain_confusion_matrix,f1_macro,f1_weighted,f1_true,f1_false
4,deepseek_test_predictions_with_experts_5_check...,deepseek,5,deepseek_k5,1263,0.661124,0.649474,0.866573,[[617 95]\n [333 218]],0.787015,1263,[[535 2 63 7 6 0]\n [ 8 99 12 0 ...,0.623554,0.638714,0.742479,0.504630
5,deepseek_test_predictions_with_experts_8_check...,deepseek,8,deepseek_k8,1263,0.609660,0.596646,0.949438,[[676 36]\n [457 94]],0.743468,1263,[[387 27 1 9 30 6 8 4]\n [ 8 116 ...,0.504428,0.533538,0.732791,0.276065
3,deepseek_test_predictions_with_experts_12_chec...,deepseek,12,deepseek_k12,1263,0.608868,0.599634,0.921348,[[656 56]\n [438 113]],0.679445,1226,[[227 16 2 3 7 0 1 2 1 9 1 ...,0.520178,0.546475,0.726467,0.313889
7,gemma_test_predictions_with_experts_5_checkabi...,gemma,5,gemma_k5,535,0.433645,0.660714,0.310924,[[111 246]\n [ 57 121]],0.775701,535,[[246 2 28 6 6 0]\n [ 3 16 6 2 ...,0.433447,0.429904,0.422857,0.444037
8,gemma_test_predictions_with_experts_8_checkabi...,gemma,8,gemma_k8,535,0.620561,0.695431,0.767507,[[274 83]\n [120 58]],0.738318,535,[[188 11 2 9 7 4 2 1]\n [ 4 53 ...,0.546665,0.607903,0.729694,0.363636
6,gemma_test_predictions_with_experts_12_checkab...,gemma,12,gemma_k12,535,0.631776,0.703046,0.775910,[[277 80]\n [117 61]],0.629559,521,[[91 20 1 3 3 0 1 3 0 5 1 2 2]\n [1...,0.560064,0.619492,0.737683,0.382445
1,Llama_test_predictions_with_experts_5_checkabi...,llama,5,llama_k5,1100,0.542727,0.752613,0.333333,[[216 432]\n [ 71 381]],0.770000,1100,[[487 3 45 4 9 2]\n [ 9 74 10 2 ...,0.532202,0.519699,0.462032,0.602372
2,Llama_test_predictions_with_experts_8_checkabi...,llama,8,llama_k8,1100,0.637273,0.682284,0.719136,[[466 182]\n [217 235]],0.743636,1100,[[350 24 2 6 29 2 4 2]\n [ 15 97 ...,0.620538,0.634737,0.700225,0.540852
0,Llama_test_predictions_with_experts_12_checkab...,llama,12,llama_k12,1100,0.633636,0.683109,0.705247,[[457 191]\n [212 240]],0.683817,1069,[[204 14 0 4 7 0 1 2 0 9 1 ...,0.618801,0.632201,0.694002,0.543601


# test

In [7]:
import argparse
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

SUPER_LABELS = [
    "economy",
    "health_social",
    "foreign_security",
    "law_rights",
    "politics_government",
    "environment_energy",
    "society_culture",
    "misc",
]

def norm_bool_label(x) -> str:
    """Normalisiert Labels robust auf 'True'/'False'."""
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return "False"
    s = str(x).strip().strip(".,:;!?)(").lower()
    if s.startswith("true"):
        return "True"
    if s.startswith("false"):
        return "False"
    return "False"


def evaluate_predictions(csv_path: str):
    df = pd.read_csv(csv_path, dtype=str)

    # Required columns
    required = ["label", "verdict_pred"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(
            f"CSV missing required columns: {missing}\n"
            f"Found columns: {list(df.columns)}"
        )

    # Clean
    df = df.dropna(subset=["label", "verdict_pred"])
    df = df[df["label"].str.strip() != ""]
    df = df[df["verdict_pred"].str.strip() != ""]

    y_true = df["label"].map(norm_bool_label)
    y_pred = df["verdict_pred"].map(norm_bool_label)

    # Overall verdict metrics
    verdict_acc = accuracy_score(y_true, y_pred)
    cm = confusion_matrix(y_true, y_pred, labels=["True", "False"])

    print("\n==============================")
    print("✅ Verdict Evaluation (overall)")
    print("==============================")
    print(f"Samples: {len(df)}")
    print(f"Verdict Accuracy: {verdict_acc:.3f}\n")

    print("Classification report (True/False):")
    print(classification_report(y_true, y_pred, labels=["True", "False"], zero_division=0))

    print("Confusion matrix [rows=true, cols=pred] (True, False):")
    print(cm)

    # Optional: Routing accuracy (if available)
    if "super_domain" in df.columns and "domain_pred" in df.columns:
        df_r = df.dropna(subset=["super_domain", "domain_pred"])
        df_r = df_r[df_r["super_domain"].str.strip() != ""]
        df_r = df_r[df_r["domain_pred"].str.strip() != ""]

        if len(df_r) > 0:
            dom_acc = accuracy_score(df_r["super_domain"], df_r["domain_pred"])
            dom_cm = confusion_matrix(df_r["super_domain"], df_r["domain_pred"], labels=SUPER_LABELS)

            print("\n==================================")
            print("🌍 Domain Routing Evaluation")
            print("==================================")
            print(f"Routing samples: {len(df_r)}")
            print(f"Domain-Routing Accuracy: {dom_acc:.3f}")
            print("Domain-Routing Confusion Matrix (rows=true, cols=pred):")
            print(dom_cm)
            print("Domain label order:", SUPER_LABELS)

    # Optional: Per-domain verdict accuracy
    if "super_domain" in df.columns:
        print("\n=================================================")
        print("📊 Per-Domain Verdict Accuracy (by true super_domain)")
        print("=================================================")

        for dom in SUPER_LABELS:
            df_dom = df[df["super_domain"] == dom]
            if df_dom.empty:
                continue
            yt = df_dom["label"].map(norm_bool_label)
            yp = df_dom["verdict_pred"].map(norm_bool_label)
            acc_dom = accuracy_score(yt, yp)
            print(f"{dom:22s} n={len(df_dom):4d} acc={acc_dom:.3f}")


In [ ]:
from multiagent_pipeline_reassurance import run_full_pipeline

SUPER_LABELS_8 = [
        "economy",
        "health_social",
        "foreign_security",
        "law_rights",
        "politics_government",
        "environment_energy",
        "society_culture",
        "misc",
]

df_pred = run_full_pipeline(
    base_model_id="meta-llama/Llama-3.1-8B-Instruct",
    router_lora_dir="adapters/llama-liar-statement-domain-lora_8Classes",
    expert_root="adapters/Llama_experts_8Classes",
    super_labels=SUPER_LABELS_8,
    test_path="Results/preprocessed_test_cleaned_binary_with_super_domain_8.csv",
    out_path="Results/Llama_test_predictions_with_experts_8_reassurance.csv",
    text_col="statement",
    label_col="label",
    domain_col="super_domain",
    sep="\t",
    max_rows=2000,                  # nur die ersten 100 Zeilen verwenden
    use_self_verification=True,    # Modell wird nach Antwort nochmal gefragt
)


ℹ️  max_rows=2000 → using first 1267 rows of the CSV.
Test set size: 1267
True domain distribution:
super_domain
economy                473
politics_government    233
law_rights             145
health_social          143
foreign_security       122
society_culture         61
environment_energy      56
misc                    34
Name: count, dtype: int64

=== Phase 1: Routing (Statement -> super_domain) ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🌍 Routing-only: 100%|██████████| 1267/1267 [10:41<00:00,  1.98it/s]



Predicted domain distribution (router):
domain_pred_router
economy                506
politics_government    201
health_social          162
law_rights             139
foreign_security       114
society_culture         65
environment_energy      57
misc                    23
Name: count, dtype: int64


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

...


  🔄 Self-verification changed verdict: False → True
     Claim: Insured Floridians pay about $2000 for every hospital stay to cover the cost of ...


🧠 Expert: health_social:  15%|█▌        | 25/162 [08:47<52:20, 22.92s/it]


  🔄 Self-verification changed verdict: True → False
     Claim: Says that each year about 25000 American women become pregnant through rape or i...


🧠 Expert: health_social:  16%|█▌        | 26/162 [09:13<53:46, 23.72s/it]


  🔄 Self-verification changed verdict: True → False
     Claim: Says Denmarks suicide rate has been about twice as high as the United States ove...


In [1]:
pip freeze > requirements_freeze.txt

Note: you may need to restart the kernel to use updated packages.


In [7]:
evaluate_predictions("Results/Llama_test_predictions_with_experts_8_reassurance.csv")


✅ Verdict Evaluation (overall)
Samples: 1267
Verdict Accuracy: 0.591

Classification report (True/False):
              precision    recall  f1-score   support

        True       0.64      0.64      0.64       714
       False       0.53      0.53      0.53       553

    accuracy                           0.59      1267
   macro avg       0.58      0.58      0.58      1267
weighted avg       0.59      0.59      0.59      1267

Confusion matrix [rows=true, cols=pred] (True, False):
[[454 260]
 [258 295]]

🌍 Domain Routing Evaluation
Routing samples: 1267
Domain-Routing Accuracy: 0.740
Domain-Routing Confusion Matrix (rows=true, cols=pred):
[[393  28   2   7  29   3   8   3]
 [ 15 108   3  13   1   0   3   0]
 [  8   3  95   9   2   2   2   1]
 [  7  13   5 106  11   1   2   0]
 [ 55   4   6   3 143  10   8   4]
 [  8   0   1   0   5  39   1   2]
 [  7   3   2   1   6   0  41   1]
 [ 13   3   0   0   4   2   0  12]]
Domain label order: ['economy', 'health_social', 'foreign_security', 

In [1]:
from multiagent_pipeline_2 import run_full_pipeline

SUPER_LABELS_8 = [
        "economy",
        "health_social",
        "foreign_security",
        "law_rights",
        "politics_government",
        "environment_energy",
        "society_culture",
        "misc",
]


df_pred = run_full_pipeline(
    base_model_id="meta-llama/Llama-3.1-8B-Instruct",
    router_lora_dir="adapters/llama-liar-statement-domain-lora_8Classes",
    expert_root="adapters/Llama_experts_8Classes",
    super_labels=SUPER_LABELS_8,
    test_path="Results/preprocessed_test_cleaned_binary_with_super_domain_8.csv",
    out_path="Results/Llama_test_predictions_with_experts_8_2.csv",
)


2026-02-20 05:43:40.967310: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Test set size: 1267
True domain distribution:
super_domain
economy                473
politics_government    233
law_rights             145
health_social          143
foreign_security       122
society_culture         61
environment_energy      56
misc                    34
Name: count, dtype: int64


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🌍 Routing-only: 100%|██████████| 1267/1267 [10:39<00:00,  1.98it/s]


...

[WARN] JSON parse failed for domain 'misc': No JSON object found in model output


🧠 misc:  75%|███████▌  | 24/32 [04:37<01:56, 14.51s/it]

[WARN] JSON parse failed for domain 'misc': No JSON object found in model output


🧠 misc: 100%|██████████| 32/32 [05:57<00:00, 11.17s/it]


🌍 Domain routing accuracy: 0.718
Confusion matrix (rows=true, cols=pred):
[[403  28   0   5  20   5   7   5]
 [ 13 110   2  12   2   0   4   0]
 [ 11   2  90  11   2   3   2   1]
 [  9  15   5 103   8   1   2   2]
 [ 69   4   5   5 117  11  12  10]
 [ 12   0   1   0   3  38   2   0]
 [ 10   2   2   5   3   0  37   2]
 [ 16   2   0   0   2   2   0  12]]
Label order: ['economy', 'health_social', 'foreign_security', 'law_rights', 'politics_government', 'environment_energy', 'society_culture', 'misc']

✅ Verdict accuracy (overall): 0.639

Classification report (True/False):
              precision    recall  f1-score   support

        True       0.68      0.67      0.68       714
       False       0.58      0.60      0.59       553

    accuracy                           0.64      1267
   macro avg       0.63      0.63      0.63      1267
weighted avg       0.64      0.64      0.64      1267

Confusion matrix [rows=true, cols=pred] (True, False):
[[477 237]
 [221 332]]

===== Per-domain

In [3]:
evaluate_predictions("Results/Llama_test_predictions_with_experts_8_2.csv")



✅ Verdict Evaluation (overall)
Samples: 1267
Verdict Accuracy: 0.639

Classification report (True/False):
              precision    recall  f1-score   support

        True       0.68      0.67      0.68       714
       False       0.58      0.60      0.59       553

    accuracy                           0.64      1267
   macro avg       0.63      0.63      0.63      1267
weighted avg       0.64      0.64      0.64      1267

Confusion matrix [rows=true, cols=pred] (True, False):
[[477 237]
 [221 332]]

🌍 Domain Routing Evaluation
Routing samples: 1267
Domain-Routing Accuracy: 0.718
Domain-Routing Confusion Matrix (rows=true, cols=pred):
[[403  28   0   5  20   5   7   5]
 [ 13 110   2  12   2   0   4   0]
 [ 11   2  90  11   2   3   2   1]
 [  9  15   5 103   8   1   2   2]
 [ 69   4   5   5 117  11  12  10]
 [ 12   0   1   0   3  38   2   0]
 [ 10   2   2   5   3   0  37   2]
 [ 16   2   0   0   2   2   0  12]]
Domain label order: ['economy', 'health_social', 'foreign_security', 

In [9]:
# =========================
# 0) Environment & Imports
# =========================
import os
os.environ.setdefault("TRANSFORMERS_NO_TF", "1")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "1")  # eine GPU auswählen

import json
import torch
import pandas as pd
from typing import Literal, Dict, Any
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    pipeline as hf_pipeline,
)
from peft import PeftModel

from langchain_experimental.pydantic_v1 import BaseModel, Field
from langchain_experimental.llms import LMFormatEnforcer

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# =========================
# 1) Konstanten & Labels
# =========================

BASE_MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"

# Router-LoRA (Statement -> super_domain)
ROUTER_LORA_DIR = "./adapters/llama-liar-statement-domain-lora_8Classes"  # <--- ANPASSEN

# Experten-LoRAs: je Domain ein Unterordner in EXPERT_ROOT
EXPERT_ROOT = "./adapters/Llama_experts_8Classes"  # <--- ANPASSEN

SUPER_LABELS = [
    "economy",
    "health_social",
    "foreign_security",
    "law_rights",
    "politics_government",
    "environment_energy",
    "society_culture",
    "misc",
]

# =========================
# 2) JSON-Schema & Helper
# =========================

class ClaimVerdict(BaseModel):
    verdict: Literal["True", "False"] = Field(..., description="Binary verdict")
    explanation: str = Field(..., description="2–4 sentences, brief, no stepwise reasoning")

JSON_SCHEMA = ClaimVerdict.schema()


def extract_first_json(text_or_obj):
    """
    Versucht robust, ein JSON-Objekt zu extrahieren.
    - Falls schon ein dict: direkt zurückgeben
    - Falls Liste mit dicts: ersten dict nehmen
    - Falls String: ersten JSON-Block via JSONDecoder finden
    """
    # Falls LMFormatEnforcer schon ein Dict zurückgibt
    if isinstance(text_or_obj, dict):
        return text_or_obj

    if isinstance(text_or_obj, list):
        # evtl. Liste von Strings / Dicts
        for el in text_or_obj:
            if isinstance(el, dict):
                return el
            if isinstance(el, str):
                try:
                    dec = json.JSONDecoder()
                    s = el.strip()
                    for i, ch in enumerate(s):
                        if ch == "{":
                            try:
                                obj, _ = dec.raw_decode(s[i:])
                                return obj
                            except json.JSONDecodeError:
                                continue
                except Exception:
                    continue

    if not isinstance(text_or_obj, str):
        raise ValueError(f"Cannot parse JSON from type {type(text_or_obj)}")

    dec = json.JSONDecoder()
    s = text_or_obj.strip()
    for i, ch in enumerate(s):
        if ch == "{":
            try:
                obj, _ = dec.raw_decode(s[i:])
                return obj
            except json.JSONDecodeError:
                continue

    # Wenn wir hier ankommen, ist wirklich kein JSON drin
    raise ValueError("No JSON object found in model output")


def norm_bool_label(x: str) -> str:
    if not isinstance(x, str):
        return "False"
    s = x.strip().strip(".,:;!?)(").lower()
    if s.startswith("true"):
        return "True"
    if s.startswith("false"):
        return "False"
    return "False"


# =========================
# 3) Router: Statement -> super_domain
# =========================

ROUTER_SYSTEM_PROMPT = """You are a strict domain classifier.

You receive a political or public policy claim (statement) and must map it
to exactly ONE high-level domain label from this list:

- economy: money, jobs, taxes, trade, business, budget, social security, pensions, stimulus
- health_social: healthcare, medicare, medicaid, public health, drugs, families, women, children, veterans, hunger, disability
- foreign_security: foreign policy, wars, military, terrorism, homeland security, nuclear weapons, international relations
- law_rights: crime, courts, civil rights, guns, immigration, legal issues, human rights, abortion, constitutional questions
- politics_government: elections, campaigns, candidates, parties, congress, political promises, redistricting, government rules and procedures
- environment_energy: climate, environment, energy, oil, gas, water, weather, transportation, pollution, natural resources
- society_culture: education, religion, diversity, LGBT, marriage, sports, pop culture, social norms, demographics
- misc: everything that does not clearly fit any of the above

You MUST answer with EXACTLY ONE label string from this list.
Do NOT explain. Only output the label.
"""


def run_router(df: pd.DataFrame) -> pd.Series:
    """
    Phase 1: Router-LoRA laden, alle Statements routen, danach Modell freigeben.
    """
    print("\n=== Phase 1: Routing (Statement -> super_domain) ===")

    # Tokenizer
    try:
        tokenizer = AutoTokenizer.from_pretrained(ROUTER_LORA_DIR, use_fast=True)
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # Base + LoRA auf *eine* GPU (oder CPU)
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
    ).to(device)

    model = PeftModel.from_pretrained(base, ROUTER_LORA_DIR)
    model = model.to(device)
    model.eval()

    import re

    preds = []

    label_list = ", ".join(SUPER_LABELS)

    for row in tqdm(df.itertuples(), total=len(df),
                   desc="🌍 Routing-only", dynamic_ncols=True):
        statement = getattr(row, "statement")

        user_content = (
            f"Valid labels:\n{label_list}\n\n"
            f"Statement:\n{statement}\n\n"
            "Return ONLY ONE label from the list above.\n"
            "Respond ONLY with the label.\n\n"
            "Label:"
        )

        messages = [
            {"role": "system", "content": ROUTER_SYSTEM_PROMPT},
            {"role": "user", "content": user_content},
        ]

        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True,
        )

        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            padding=True,
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            gen = model.generate(
                input_ids=inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                pad_token_id=tokenizer.pad_token_id,
                max_new_tokens=8,
                do_sample=False,
            )

        new_tokens = gen[0][inputs["input_ids"].shape[1]:]
        answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()

        if not answer:
            preds.append("misc")
            continue

        # 1) exakte Wort-Matches
        pred_dom = "misc"
        for L in SUPER_LABELS:
            pattern = r"\b" + re.escape(L.lower()) + r"\b"
            if re.search(pattern, answer):
                pred_dom = L
                break
        else:
            # 2) substring fallback
            for L in SUPER_LABELS:
                if L.lower() in answer:
                    pred_dom = L
                    break

        preds.append(pred_dom)

    # Router-Modell freigeben
    del model
    del base
    torch.cuda.empty_cache()

    return pd.Series(preds, index=df.index, name="domain_pred_router")


# =========================
# 4) Experten + LMFormatEnforcer
# =========================

def build_expert_system_prompt(domain_name: str) -> str:
    return f"""You are a fact-checking expert specialized in the '{domain_name}' domain.

You receive a political or public policy claim (statement) and must decide if it is factually correct.

You must output a JSON object that follows this schema:

{json.dumps(JSON_SCHEMA, indent=2)}

The field "verdict" MUST be either "True" or "False".
The field "explanation" MUST be 2–4 concise sentences. 
Do NOT include step-by-step reasoning or lists.

Reply with JSON only. No markdown, no extra text.
"""


def run_expert_for_domain(df: pd.DataFrame, domain_name: str):
    """
    Phase 2 (pro Domain):
    - Experten-LoRA für 'domain_name' laden
    - alle Zeilen dieser Domain fact-checken (mit LMFormatEnforcer)
    - Modell danach freigeben
    """
    expert_dir = os.path.join(EXPERT_ROOT, domain_name)
    if not os.path.isdir(expert_dir):
        raise FileNotFoundError(f"Expert directory for domain '{domain_name}' not found: {expert_dir}")

    print(f"\n=== Experten-Fact-Check für Domain '{domain_name}' (n={len(df)}) ===")

    # Tokenizer
    try:
        tokenizer = AutoTokenizer.from_pretrained(expert_dir, use_fast=True)
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    # Base + LoRA
    base = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
    ).to(device)

    model = PeftModel.from_pretrained(base, expert_dir)
    model = model.to(device)
    model.eval()

    # HF-Pipeline
    gen_pipe = hf_pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        do_sample=False,
        return_full_text=False,
    )

    # LMFormatEnforcer
    enforced_llm = LMFormatEnforcer(
        pipeline=gen_pipe,
        json_schema=JSON_SCHEMA,
    )

    def fact_check_one(statement: str) -> Dict[str, Any]:
        """
        Ruft den Experten + LMFormatEnforcer auf und gibt IMMER ein dict mit
        verdict ("True"/"False") und explanation (str) zurück.
        Jede Art von Parsing-/Enforcer-Fehler wird abgefangen.
        """
        system_prompt = build_expert_system_prompt(domain_name)
        user_content = f"Claim:\n{statement}\n\nReturn the JSON object now."

        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_content},
        ]

        try:
            prompt_text = tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )
        except Exception:
            prompt_text = (
                f"[SYSTEM]\n{system_prompt}\n[/SYSTEM]\n"
                f"User:\n{user_content}\nAssistant (JSON only):"
            )

        try:
            raw = enforced_llm.invoke(prompt_text, max_new_tokens=192)
            parsed = extract_first_json(raw)
            verdict = parsed.get("verdict")
            expl = parsed.get("explanation", "").strip()
        except Exception as e:
            # Hier landen wir, wenn entweder invoke oder JSON-Parsing schiefgeht
            print(f"[WARN] JSON parse failed for domain '{domain_name}': {e}")
            verdict = "False"
            expl = f"Fallback due to parsing error: {str(e)[:200]}"

        if verdict not in ["True", "False"]:
            verdict = "False"
        if not expl:
            expl = "No explanation was provided."

        return {"verdict": verdict, "explanation": expl}

    verdicts = []
    explanations = []

    for row in tqdm(df.itertuples(), total=len(df),
                   desc=f"🧠 {domain_name}", dynamic_ncols=True):
        statement = getattr(row, "statement")
        res = fact_check_one(statement)
        verdicts.append(norm_bool_label(res["verdict"]))
        explanations.append(res["explanation"])

    # Modell freigeben
    del model
    del base
    torch.cuda.empty_cache()

    return pd.Series(verdicts, index=df.index), pd.Series(explanations, index=df.index)


# =========================
# 5) Test-CSV einlesen
# =========================

test_path = "Results/preprocessed_test_cleaned_binary_with_super_domain_8.csv"  # <--- ANPASSEN

df_test = pd.read_csv(
    test_path,
    sep="\t",
    quotechar='"',
    engine="python",
    dtype=str,
)

df_test = df_test.dropna(subset=["statement", "label"])
df_test = df_test[df_test["statement"].str.strip() != ""]
df_test = df_test[df_test["label"].isin(["True", "False"])]

print("Testset size:", len(df_test))
if "super_domain" in df_test.columns:
    print("Super-Domain-Verteilung (true):")
    print(df_test["super_domain"].value_counts())


# =========================
# 6) Phase 1: Router laufen lassen
# =========================

df_test["domain_pred_router"] = run_router(df_test)

print("\nRouting-only Verteilung:")
print(df_test["domain_pred_router"].value_counts())


# =========================
# 7) Phase 2: Experten pro Domain
# =========================

df_test["domain_pred"] = df_test["domain_pred_router"]
df_test["verdict_pred"] = "False"
df_test["explanation_pred"] = ""

for dom in SUPER_LABELS:
    idxs = df_test.index[df_test["domain_pred_router"] == dom].tolist()
    if not idxs:
        print(f"\n⚠️ Keine vom Router zugewiesenen Beispiele für Domain '{dom}'.")
        continue

    df_dom = df_test.loc[idxs]
    v_dom, e_dom = run_expert_for_domain(df_dom, dom)

    df_test.loc[idxs, "verdict_pred"] = v_dom
    df_test.loc[idxs, "explanation_pred"] = e_dom


# =========================
# 8) Metriken
# =========================

print("\nBeispiel-Zeilen mit Predictions:")
cols_show = ["statement", "super_domain", "domain_pred", "label", "verdict_pred"]
print(df_test[[c for c in cols_show if c in df_test.columns]].head())

# 8a) Domain-Routing-Accuracy
if "super_domain" in df_test.columns:
    true_dom = df_test["super_domain"]
    pred_dom = df_test["domain_pred"]
    dom_acc = accuracy_score(true_dom, pred_dom)
    print(f"\n🌍 Domain-Routing Accuracy: {dom_acc:.3f}")
    print("Domain-Routing Confusion Matrix (rows=true, cols=pred):")
    print(confusion_matrix(true_dom, pred_dom, labels=SUPER_LABELS))
    print("Domain label order:", SUPER_LABELS)

# 8b) Verdict-Accuracy gesamt
y_true = df_test["label"].map(norm_bool_label)
y_pred = df_test["verdict_pred"].map(norm_bool_label)

verdict_acc = accuracy_score(y_true, y_pred)
print(f"\n✅ Verdict Accuracy (gesamt): {verdict_acc:.3f}\n")

print("Classification report (True/False):")
print(classification_report(y_true, y_pred, labels=["True", "False"], zero_division=0))

print("Confusion matrix [rows=true, cols=pred] (True, False):")
print(confusion_matrix(y_true, y_pred, labels=["True", "False"]))

# 8c) Per-Domain Verdict-Accuracy
if "super_domain" in df_test.columns:
    print("\n===== Per-Domain Verdict-Accuracy (nach true super_domain) =====")
    for dom in SUPER_LABELS:
        df_dom_true = df_test[df_test["super_domain"] == dom]
        if df_dom_true.empty:
            continue
        yt = df_dom_true["label"].map(norm_bool_label)
        yp = df_dom_true["verdict_pred"].map(norm_bool_label)
        acc_dom = accuracy_score(yt, yp)
        print(f"{dom:22s} n={len(df_dom_true):4d} acc={acc_dom:.3f}")


# =========================
# 9) Ergebnisse speichern
# =========================

out_path = "Results/test_predictions_with_experts_sequential.csv"  # <--- ANPASSEN
os.makedirs(os.path.dirname(out_path), exist_ok=True)
df_test.to_csv(out_path, index=False, encoding="utf-8")
print(f"\n💾 Predictions gespeichert unter:\n{out_path}")


Using device: cuda
Testset size: 1267
Super-Domain-Verteilung (true):
super_domain
economy                473
politics_government    233
law_rights             145
health_social          143
foreign_security       122
society_culture         61
environment_energy      56
misc                    34
Name: count, dtype: int64

=== Phase 1: Routing (Statement -> super_domain) ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🌍 Routing-only: 100%|██████████| 1267/1267 [10:21<00:00,  2.04it/s]



Routing-only Verteilung:
domain_pred_router
economy                543
health_social          163
politics_government    157
law_rights             141
foreign_security       105
society_culture         66
environment_energy      60
misc                    32
Name: count, dtype: int64

=== Experten-Fact-Check für Domain 'economy' (n=543) ===


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

...

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Device set to use cuda:0
🧠 misc:  75%|███████▌  | 24/32 [03:47<01:46, 13.29s/it]

[WARN] JSON parse failed for domain 'misc': No JSON object found in model output


🧠 misc: 100%|██████████| 32/32 [05:02<00:00,  9.47s/it]



Beispiel-Zeilen mit Predictions:
                                           statement   super_domain  \
0  Building a wall on the U.S . Mexico border wil...     law_rights   
1  Wisconsin is on pace to double the number of l...        economy   
2  Says John McCain has done nothing to help the ...  health_social   
3  Suzanne Bonamici supports a plan that will cut...  health_social   
4  When asked by a reporter whether hes at the ce...     law_rights   

     domain_pred  label verdict_pred  
0     law_rights   True         True  
1        economy  False         True  
2  health_social  False        False  
3  health_social   True         True  
4        economy  False        False  

🌍 Domain-Routing Accuracy: 0.718
Domain-Routing Confusion Matrix (rows=true, cols=pred):
[[403  28   0   5  20   5   7   5]
 [ 13 110   2  12   2   0   4   0]
 [ 11   2  90  11   2   3   2   1]
 [  9  15   5 103   8   1   2   2]
 [ 69   4   5   5 117  11  12  10]
 [ 12   0   1   0   3  38   2   0]
 [ 10

# old code

In [16]:
import os
import torch
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

os.environ.setdefault("CUDA_VISIBLE_DEVICES", "7")

BASE_MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
EXPERT_ROOT = "adapters/Llama_experts_8Classes"  # da liegen expert_<domain> drin


SUPER_LABELS = [
    "economy",
    "health_social",
    "foreign_security",
    "law_rights",
    "politics_government",
    "environment_energy",
    "society_culture",
    "misc",
]


In [17]:
def build_expert_system_prompt(domain_name: str) -> str:
    return f"""You are a fact-checking expert specialized in the '{domain_name}' domain.

You receive short political or public policy claims (statements) and must decide if they are factually correct.

Answer STRICTLY with one of these two labels:
- True
- False

Do NOT explain. Only output 'True' or 'False'.
"""


def load_expert_model(domain_name: str):
    """
    Lädt Base LLaMA + passenden LoRA-Adapter für die Domain.
    """
    expert_dir = os.path.join(EXPERT_ROOT, f"{domain_name}")
    if not os.path.isdir(expert_dir):
        raise FileNotFoundError(f"Expert directory not found: {expert_dir}")

    # Base-Modell
    base_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
        device_map="auto",
    )

    # LoRA-Adapter drauf
    model = PeftModel.from_pretrained(base_model, expert_dir)
    model.eval()

    tokenizer = AutoTokenizer.from_pretrained(expert_dir, use_fast=True)
    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    if torch.cuda.is_available():
        model = model.to("cuda")

    return model, tokenizer


def predict_with_expert(statement: str, domain_name: str, model, tokenizer) -> str:
    """
    Nutzt ein geladenes Expertenmodell, um für ein Statement True/False zu predicten.
    """
    system_prompt = build_expert_system_prompt(domain_name)
    user_content = f"Claim:\n{statement}\n\nAnswer:"

    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_content},
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        padding=True,
    )

    if torch.cuda.is_available():
        inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        gen = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            pad_token_id=tokenizer.pad_token_id,
            max_new_tokens=4,
            do_sample=False,
        )

    new_tokens = gen[0][inputs["input_ids"].shape[1]:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    # Normalisieren
    first = answer.split()[0].strip().strip(".,:;!?)(").lower() if answer else ""

    if first.startswith("true"):
        return "True"
    if first.startswith("false"):
        return "False"

    # Fallback
    return "False"


In [18]:
test_path = "../own_datasets/test_binary_labels_with_super_domain.csv"

df_test = pd.read_csv(
    test_path,
    sep="\t",
    quotechar='"',
    engine="python",
    dtype=str,
)

# Aufräumen
df_test = df_test.dropna(subset=["statement", "label", "super_domain"])
df_test = df_test[df_test["statement"].str.strip() != ""]
df_test = df_test[df_test["label"].isin(["True", "False"])]

print("Testset size:", len(df_test))
print("Super-Domain-Verteilung (Test):")
print(df_test["super_domain"].value_counts())


Testset size: 1267
Super-Domain-Verteilung (Test):
super_domain
economy                473
politics_government    233
law_rights             145
health_social          143
foreign_security       122
society_culture         61
environment_energy      56
misc                    34
Name: count, dtype: int64


In [19]:
all_results = []
all_rows_with_pred = []  # später für Gesamtauswertung

for domain in SUPER_LABELS:
    df_dom = df_test[df_test["super_domain"] == domain].copy()
    if df_dom.empty:
        print(f"\n⚠️ Domain '{domain}': keine Testbeispiele – wird übersprungen.")
        continue

    print(f"\n===== Evaluating expert for domain: {domain} =====")
    print(f"# Testbeispiele: {len(df_dom)}")
    print("Label-Verteilung (true):")
    print(df_dom["label"].value_counts())

    # Modell laden
    model, tok = load_expert_model(domain)

    preds = []
    for _, row in tqdm(df_dom.iterrows(), total=len(df_dom), desc=f"🧠 {domain}", dynamic_ncols=True):
        statement = row["statement"]
        pred = predict_with_expert(statement, domain, model, tok)
        preds.append(pred)

    df_dom["pred"] = preds
    all_rows_with_pred.append(df_dom)

    y_true = df_dom["label"]
    y_pred = df_dom["pred"]

    acc = accuracy_score(y_true, y_pred)
    print(f"\nAccuracy ({domain}): {acc:.3f}")

    report = classification_report(y_true, y_pred, labels=["True", "False"], zero_division=0)
    print("Classification report:")
    print(report)

    cm = confusion_matrix(y_true, y_pred, labels=["True", "False"])
    print("Confusion matrix [rows=true, cols=pred] (True, False):")
    print(cm)

    all_results.append({
        "domain": domain,
        "n": len(df_dom),
        "accuracy": acc,
    })

    # Speicher freigeben
    del model
    torch.cuda.empty_cache()



===== Evaluating expert for domain: economy =====
# Testbeispiele: 473
Label-Verteilung (true):
label
True     266
False    207
Name: count, dtype: int64


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🧠 economy: 100%|██████████| 473/473 [01:28<00:00,  5.35it/s]



Accuracy (economy): 0.670
Classification report:
              precision    recall  f1-score   support

        True       0.70      0.72      0.71       266
       False       0.63      0.61      0.62       207

    accuracy                           0.67       473
   macro avg       0.66      0.66      0.66       473
weighted avg       0.67      0.67      0.67       473

Confusion matrix [rows=true, cols=pred] (True, False):
[[191  75]
 [ 81 126]]

===== Evaluating expert for domain: health_social =====
# Testbeispiele: 143
Label-Verteilung (true):
label
True     82
False    61
Name: count, dtype: int64


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

...

🧠 society_culture: 100%|██████████| 61/61 [00:09<00:00,  6.77it/s]



Accuracy (society_culture): 0.574
Classification report:
              precision    recall  f1-score   support

        True       0.67      0.59      0.63        37
       False       0.46      0.54      0.50        24

    accuracy                           0.57        61
   macro avg       0.57      0.57      0.56        61
weighted avg       0.59      0.57      0.58        61

Confusion matrix [rows=true, cols=pred] (True, False):
[[22 15]
 [11 13]]

===== Evaluating expert for domain: misc =====
# Testbeispiele: 34
Label-Verteilung (true):
label
True     23
False    11
Name: count, dtype: int64


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

🧠 misc: 100%|██████████| 34/34 [00:06<00:00,  5.19it/s]


Accuracy (misc): 0.647
Classification report:
              precision    recall  f1-score   support

        True       0.82      0.61      0.70        23
       False       0.47      0.73      0.57        11

    accuracy                           0.65        34
   macro avg       0.65      0.67      0.64        34
weighted avg       0.71      0.65      0.66        34

Confusion matrix [rows=true, cols=pred] (True, False):
[[14  9]
 [ 3  8]]


In [20]:
df_all_pred = pd.concat(all_rows_with_pred, ignore_index=True)

print("\n===== Gesamt-Performance über alle Domains =====")
y_true_all = df_all_pred["label"]
y_pred_all = df_all_pred["pred"]

acc_all = accuracy_score(y_true_all, y_pred_all)
print(f"Overall accuracy: {acc_all:.3f}")

print("Overall classification report:")
print(classification_report(y_true_all, y_pred_all, labels=["True", "False"], zero_division=0))

print("Overall confusion matrix (True, False):")
print(confusion_matrix(y_true_all, y_pred_all, labels=["True", "False"]))

# Optional: pro Domain Accuracy-Tabelle
print("\nPer-domain accuracy summary:")
for r in all_results:
    print(f"{r['domain']:20s}  n={r['n']:4d}  acc={r['accuracy']:.3f}")



===== Gesamt-Performance über alle Domains =====
Overall accuracy: 0.654
Overall classification report:
              precision    recall  f1-score   support

        True       0.69      0.71      0.70       714
       False       0.61      0.58      0.59       553

    accuracy                           0.65      1267
   macro avg       0.65      0.65      0.65      1267
weighted avg       0.65      0.65      0.65      1267

Overall confusion matrix (True, False):
[[510 204]
 [234 319]]

Per-domain accuracy summary:
economy               n= 473  acc=0.670
health_social         n= 143  acc=0.629
foreign_security      n= 122  acc=0.689
law_rights            n= 145  acc=0.628
politics_government   n= 233  acc=0.670
environment_energy    n=  56  acc=0.607
society_culture       n=  61  acc=0.574
misc                  n=  34  acc=0.647


# Chat-Template

In [2]:
from transformers import AutoTokenizer

for mid in [
    "meta-llama/Llama-3.1-8B-Instruct",
    "google/gemma-7b-it",
    "deepseek-ai/deepseek-llm-7b-base",
]:
    tok = AutoTokenizer.from_pretrained(mid, trust_remote_code=True)
    has_chat = hasattr(tok, "apply_chat_template") and bool(getattr(tok, "chat_template", None))
    print(mid, "has_chat=", has_chat, "chat_template_present=", bool(getattr(tok, "chat_template", None)))
    if has_chat:
        print("template snippet:", tok.chat_template[:200], "...\n")

meta-llama/Llama-3.1-8B-Instruct has_chat= True chat_template_present= True
template snippet: {{- bos_token }}
{%- if custom_tools is defined %}
    {%- set tools = custom_tools %}
{%- endif %}
{%- if not tools_in_user_message is defined %}
    {%- set tools_in_user_message = true %}
{%- endif ...

google/gemma-7b-it has_chat= True chat_template_present= True
template snippet: {{ bos_token }}{% if messages[0]['role'] == 'system' %}{{ raise_exception('System role not supported') }}{% endif %}{% for message in messages %}{% if (message['role'] == 'user') != (loop.index0 % 2 = ...

deepseek-ai/deepseek-llm-7b-base has_chat= False chat_template_present= False
